<a href="https://colab.research.google.com/github/ahmedyomiisiaka/RP-Network-Analysis/blob/main/notebooks/05_Context_Specific_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/data/network",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

drive_path = Path("/content/drive/MyDrive/rp_network")
colab_path = Path("/content/rp_network")

for f in drive_path.rglob("*"):
    if f.is_file():
        dest = colab_path / f.relative_to(drive_path)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)

print("✓ All files restored from Google Drive.")
print()
for f in sorted(colab_path.rglob("*.tsv")):
    print(f"  {f.relative_to(colab_path)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ All files restored from Google Drive.

  data/processed/compartments_interaction_annotation_t2.tsv
  data/processed/compartments_interaction_annotation_t3.tsv
  data/processed/compartments_shared_t2.tsv
  data/processed/context_specific_evidence.tsv
  data/processed/context_specific_evidence_master.tsv
  data/processed/filter_A_compartments.tsv
  data/processed/filter_A_passed.tsv
  data/processed/filter_B_expression.tsv
  data/processed/filter_B_passed.tsv
  data/processed/hpa_retina_interaction_annotation.tsv
  data/processed/network_filtered.tsv
  data/processed/ppi_interactions.tsv
  data/processed/ppi_nodes.tsv
  data/raw/compartments_human.tsv
  data/raw/rna_tissue_consensus.tsv
  data/raw/string_checkpoint.tsv
  data/raw/string_interactions_raw.tsv


# Notebook 05 — Context-Specific Network Filtering

**Project:** Network-Based Analysis of Signalling Pathways in Retinitis Pigmentosa
**Institution:** Vilnius University, Systems Biology
**Supervisor:** Doc. Dr. Erinija Pranckevičienė
**Author:** Ahmed Yomi Isiaka
**Stage:** 5 — Context-Specific Network Filtering
**Date:** 2026-08-13

---

## 1. Objective

To filter the raw PPI network from Stage 4 to retain only
interactions that are biologically plausible in the retinal
photoreceptor context.

The Stage 4 network contains 9,835 interactions among 4,132
proteins retrieved at STRING confidence score ≥ 0.400. Not all
of these interactions are active in retinal tissue — many are
generic interactions documented in non-retinal cell types or
subcellular compartments that do not co-occur in photoreceptors.

This notebook applies two sequential biological filters:

**Filter 1 — Subcellular co-localisation (COMPARTMENTS)**
Both interacting proteins must be present in the same
subcellular compartment. Proteins in different compartments
cannot physically interact.

**Filter 2 — Retinal tissue expression (Human Protein Atlas)**
Both interacting proteins must be expressed in retinal tissue.
Proteins not expressed in the retina are not relevant to
retinal disease biology.

An interaction passes the filter only if it satisfies
BOTH criteria.

## Supervisor guidance (2026-08-13)

> "Please focus on what we were discussing — databases to filter
> interactions: COMPARTMENTS and other. The lowest threshold is
> good since we will have to retain interactions that are in the
> same compartment and tissue."

This confirms:
- Working network: score ≥ 0.400 (9,835 edges, 4,132 proteins)
- Filter 1: COMPARTMENTS database — subcellular co-localisation
- Filter 2: Human Protein Atlas — retinal tissue expression

---
## 2. Background and Rationale

### Why context-specific filtering is necessary

Protein-protein interaction databases such as STRING aggregate
interaction evidence from many cell types, tissues, and
experimental conditions. A high STRING score means an interaction
is well-supported by evidence — but that evidence may come from
liver cells, cancer cell lines, or yeast two-hybrid experiments
that do not reflect what happens in a retinal photoreceptor.

For this project, we are specifically interested in:
- Interactions that occur in retinal photoreceptor cells
- Proteins that are physically co-located (same compartment)
- Proteins that are actually expressed in retinal tissue

Filtering by these biological criteria produces a smaller,
higher-quality network that is specific to retinal RP biology
rather than generic human cell biology.

### The two filters

**Filter 1 — Subcellular co-localisation**
Physical protein-protein interaction requires that both proteins
are present in the same subcellular compartment at the same time.
For example, a membrane protein cannot directly interact with a
strictly nuclear protein. The COMPARTMENTS database provides
confidence-scored localisation data for human proteins across
all major subcellular compartments.

**Filter 2 — Retinal tissue expression**
Only proteins expressed in retinal tissue are relevant to RP
biology. The Human Protein Atlas provides RNA expression data
across 54 human tissues including the retina, scored as:
Not detected / Low / Medium / High / Enhanced.
We retain only proteins with detectable retinal expression.

### Filter order

Filter 1 (COMPARTMENTS) is applied first because:
- It is a direct biological constraint on physical interaction
- It does not depend on disease state — a protein in the nucleus
  cannot interact with a membrane protein regardless of tissue
- COMPARTMENTS data is downloadable as a flat file — efficient

Filter 2 (Human Protein Atlas) is applied to the output of
Filter 1 — a smaller network, making the query faster.

---
## 4. Methodological Notes — All Databases Used

---

### Database 1 — COMPARTMENTS (Jensen Lab)

**What it is:**
COMPARTMENTS is a weekly-updated database of protein subcellular
localisation evidence integrating experimental data, curated
databases (UniProt, Gene Ontology), text mining, and sequence-
based prediction. Each protein-compartment pair is assigned a
confidence score from 1 (weak) to 5 (strong).

**How it was queried:**
Downloaded as a flat TSV file from:
https://download.jensenlab.org/human_compartment_integrated_full.tsv
Access date: 2026-08-13
No API call required — direct file download.

**What information was retrieved:**
For each human protein: which subcellular compartments it
localises to, and the confidence score for each localisation.

**How it was used:**
For each interaction edge (protein A — protein B):
- Retrieved all compartments for protein A (score ≥ 3)
- Retrieved all compartments for protein B (score ≥ 3)
- If the two compartment sets overlap → edge RETAINED
- If no overlap exists → proteins cannot co-localise → edge REMOVED

**Score threshold:** ≥ 3 (medium-high confidence)

**Citation:**
Binder JX, et al. (2014). COMPARTMENTS: unification and
visualization of protein subcellular localization evidence.
Database, bau012. https://doi.org/10.1093/database/bau012

---

### Database 2 — Human Protein Atlas (HPA)

**What it is:**
The Human Protein Atlas provides RNA and protein expression data
across 54 human tissues including the retina, combining RNA
sequencing (nTPM values) and antibody-based protein profiling.

**How it was queried:**
Downloaded as a flat TSV file from:
https://www.proteinatlas.org/about/download
File: rna_tissue_consensus.tsv
Access date: 2026-08-13
Contains nTPM values for all human genes across all tissues.

**What information was retrieved:**
For each protein: nTPM (normalised transcripts per million)
expression value specifically in retinal tissue.

**How it was used:**
For each protein in the network after COMPARTMENTS filtering:
- Retrieved nTPM value in retina
- nTPM > 0 → protein expressed in retina → RETAINED
- nTPM = 0 or not found → not expressed → edge FLAGGED
An edge is kept only if BOTH proteins have retinal expression.

**Citation:**
Uhlén M, et al. (2015). Tissue-based map of the human proteome.
Science, 347(6220):1260419.
https://doi.org/10.1126/science.1260419

---

### Database 3 — IID (Integrated Interactions Database)

**What it is:**
IID provides tissue-specific protein-protein interaction data for
human and model organisms. Interactions are assigned to tissues
where their protein pairs or encoding genes are expressed.
It covers 30 tissues in humans including retina.

**How it was queried:**
Queried via the IID web interface at:
https://iid.ophid.utoronto.ca
Input: protein list, Species: Homo sapiens, Tissue: Retina
Access date: 2026-08-13

**What information was retrieved:**
Which interactions in our network have documented evidence
of activity in retinal tissue specifically.

**How it was used:**
For each edge remaining after Filters A and B:
- Checked whether IID reports this interaction as retina-active
- IID-supported → edge annotated as "retina-confirmed"
- Not in IID → edge annotated as "retina-unconfirmed"
Both are retained but annotation allows downstream filtering.

**Citation:**
Kotlyar M, et al. (2019). IID 2018 update: context-specific
physical protein-protein interactions. Nucleic Acids Res,
47(D1):D581-D589. https://doi.org/10.1093/nar/gky1037

---

### Database 4 — TissueNet v3

**What it is:**
TissueNet provides tissue-resolved scores for human PPI networks
by integrating gene expression data from multiple sources to
assign each interaction a tissue-specific activity score.

**How it was queried:**
Downloaded from: https://netbio.bgu.ac.il/tissuenet
Access date: 2026-08-13
File contains per-tissue interaction scores for human PPIs.

**What information was retrieved:**
For each interaction: a retinal tissue-specific activity score.

**How it was used:**
Used as additional evidence to score/rank interactions by their
retinal tissue relevance. Higher TissueNet retinal score →
higher confidence the interaction is active in the retina.

**Citation:**
Basha O, et al. (2015). TissueNet: The database of human tissue
protein-protein interactions. Nucleic Acids Res, 43(D1):D424-D429.

---

### Comparison of Databases

| Database | Type | Retina-specific | Format | Used for |
|----------|------|----------------|--------|----------|
| COMPARTMENTS | Subcellular localisation | No (all compartments) | TSV download | Filter A |
| Human Protein Atlas | Tissue expression | Yes (retina nTPM) | TSV download | Filter B |
| IID | Tissue-specific PPI | Yes (retina PPIs) | Web query | Filter C |
| TissueNet v3 | Tissue-resolved PPI scores | Yes (retina score) | TSV download | Filter D |

**Why we use all four:**
Each database addresses a different biological question.
COMPARTMENTS tells us IF proteins can physically meet.
HPA tells us IF proteins are present in retinal tissue.
IID tells us IF this specific interaction occurs in retina.
TissueNet tells us HOW STRONGLY the interaction is supported
in retinal tissue. Together they provide a multi-evidence
filter that is much more reliable than any single database.

---
## 5. Methodological Decisions

| Item | Decision | Status |
|------|----------|--------|
| Input network | ppi_interactions.tsv (9,835 edges) | Confirmed |
| STRING threshold | 0.400 (supervisor confirmed 2026-08-13) | Confirmed |
| Filter A | COMPARTMENTS — subcellular co-localisation | Confirmed |
| Filter B | Human Protein Atlas — retinal nTPM expression | Confirmed |
| Filter C | IID — retina-specific interaction evidence | Confirmed |
| Filter D | TissueNet v3 — retinal interaction scoring | Confirmed |
| COMPARTMENTS score cutoff | ≥ 3 (medium-high confidence) | Proposed |
| HPA expression cutoff | nTPM > 0 (any detectable expression) | Proposed |
| Filter order | A → B → C → D | Confirmed |
| Edge retention rule | Must pass Filter A AND Filter B | Confirmed |
| IID/TissueNet role | Annotation and ranking, not hard removal | Proposed |
| Seed protein handling | Seed proteins retained regardless | To confirm |

---

## 6. Output Files

| File | Description |
|------|-------------|
| `data/raw/compartments_human.tsv` | Raw COMPARTMENTS download |
| `data/raw/hpa_rna_retina.tsv` | HPA retinal expression data |
| `data/processed/filter_A_compartments.tsv` | Edges passing compartment filter |
| `data/processed/filter_B_expression.tsv` | Edges passing expression filter |
| `data/processed/filter_C_iid.tsv` | IID retinal interaction annotations |
| `data/processed/filter_D_tissuenet.tsv` | TissueNet retinal scores |
| `data/processed/network_filtered.tsv` | Final filtered interaction table |
| `data/processed/network_filtered_nodes.tsv` | Final filtered node table |
| `data/network/rp_network_filtered.graphml` | Final filtered network |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 1: Install libraries and create folders
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "requests", "pandas", "networkx"],
    check=True
)

from pathlib import Path

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/data/network",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Libraries installed.")
print("✓ Folder structure ready.")

✓ Libraries installed.
✓ Folder structure ready.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 2: Imports
# ─────────────────────────────────────────────────────────────────

import requests
import pandas as pd
import networkx as nx
import time
from pathlib import Path
from datetime import datetime

print(f"pandas   : {pd.__version__}")
print(f"networkx : {nx.__version__}")
print(f"requests : {requests.__version__}")
print("✓ All imports successful.")

pandas   : 2.2.2
networkx : 3.6.1
requests : 2.32.4
✓ All imports successful.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 3: Load the Stage 4 interaction network
#
# Input : data/processed/ppi_interactions.tsv
#         9,835 unique edges, 4,132 proteins
#         STRING confidence score >= 0.400
# ─────────────────────────────────────────────────────────────────

INTERACTION_FILE = "/content/rp_network/data/processed/ppi_interactions.tsv"
NODE_FILE        = "/content/rp_network/data/processed/ppi_nodes.tsv"

df_interactions = pd.read_csv(INTERACTION_FILE, sep="\t")
df_nodes        = pd.read_csv(NODE_FILE, sep="\t")

# Rebuild seed gene set
seed_genes = df_nodes[df_nodes["node_type"] == "Seed"]["protein_name"].tolist()
seed_upper = set(g.upper() for g in seed_genes)

print("=" * 55)
print("STAGE 4 NETWORK — LOADED")
print("=" * 55)
print(f"Interaction edges : {len(df_interactions)}")
print(f"Total proteins    : {len(df_nodes)}")
print(f"  Seed proteins   : {len(seed_genes)}")
print(f"  New partners    : {len(df_nodes) - len(seed_genes)}")
print()
print("Score distribution (confirming threshold = 0.400):")
bins = [(0.4,0.5,"0.400-0.499"), (0.5,0.7,"0.500-0.699"),
        (0.7,0.9,"0.700-0.899"), (0.9,1.01,"0.900-1.000")]
for lo, hi, label in bins:
    n = ((df_interactions["score"] >= lo) &
         (df_interactions["score"] < hi)).sum()
    print(f"  {label} : {n} edges")
print()
print("✓ Network loaded and ready for filtering.")

STAGE 4 NETWORK — LOADED
Interaction edges : 9835
Total proteins    : 4132
  Seed proteins   : 105
  New partners    : 4027

Score distribution (confirming threshold = 0.400):
  0.400-0.499 : 3590 edges
  0.500-0.699 : 3410 edges
  0.700-0.899 : 1635 edges
  0.900-1.000 : 1200 edges

✓ Network loaded and ready for filtering.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 4: Download COMPARTMENTS database
#
# Database : COMPARTMENTS (Jensen Lab, University of Copenhagen)
# File     : human_compartment_integrated_full.tsv
# URL      : https://download.jensenlab.org/
# Access   : 2026-08-13
# Size     : ~200MB — download takes 1-3 minutes in Colab
#
# File format (tab-separated):
#   Col 1: Protein identifier (Ensembl gene ID or gene name)
#   Col 2: Protein name (gene symbol)
#   Col 3: GO term ID (compartment)
#   Col 4: GO term name (compartment name)
#   Col 5: Evidence score (1-5, higher = more confident)
#   Col 6: Evidence source (knowledge, experiments, textmining, predictions)
#
# Score interpretation:
#   5 = strongest evidence (manually curated, multiple sources)
#   4 = strong evidence
#   3 = medium-high evidence ← our cutoff
#   2 = medium evidence
#   1 = weak evidence (single text mining mention)
#
# We use score >= 3 to retain well-supported localisations
# while excluding weak or single-source predictions.
# ─────────────────────────────────────────────────────────────────

COMPARTMENTS_URL  = "https://download.jensenlab.org/human_compartment_integrated_full.tsv"
COMPARTMENTS_FILE = "/content/rp_network/data/raw/compartments_human.tsv"
COMPARTMENTS_SCORE_CUTOFF = 3

print("Downloading COMPARTMENTS database...")
print(f"URL  : {COMPARTMENTS_URL}")
print(f"File : {COMPARTMENTS_FILE}")
print("This may take 1-3 minutes...")
print()

r = requests.get(COMPARTMENTS_URL, timeout=300, stream=True)
print(f"Status : {r.status_code}")

if r.status_code == 200:
    with open(COMPARTMENTS_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    size_mb = Path(COMPARTMENTS_FILE).stat().st_size / 1_048_576
    print(f"✓ Downloaded: {size_mb:.1f} MB")
else:
    print(f"✗ Error: {r.status_code} — {r.text[:200]}")

URL  : https://download.jensenlab.org/human_compartment_integrated_full.tsv
File : /content/rp_network/data/raw/compartments_human.tsv
This may take 1-3 minutes...

Status : 200
✓ Downloaded: 288.4 MB


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 5: Parse COMPARTMENTS database
#
# Load the COMPARTMENTS file and filter for:
#   - Human proteins only (already human-specific file)
#   - Score >= COMPARTMENTS_SCORE_CUTOFF (>= 3)
#
# Build a lookup dictionary:
#   protein_name → set of compartment GO terms
#
# This dictionary will be used to check compartment overlap
# for each interaction edge.
# ─────────────────────────────────────────────────────────────────

print("Loading COMPARTMENTS database...")
print(f"Score cutoff: >= {COMPARTMENTS_SCORE_CUTOFF}")
print()

# Load — no header in this file
df_comp = pd.read_csv(
    COMPARTMENTS_FILE,
    sep="\t",
    header=None,
    names=["ensembl_id", "gene_symbol", "go_term_id",
           "go_term_name", "score", "evidence_source"],
    dtype={"score": float},
    low_memory=False
)

print(f"Total rows in COMPARTMENTS file : {len(df_comp):,}")
print(f"Unique proteins                 : {df_comp['gene_symbol'].nunique():,}")
print(f"Unique compartments             : {df_comp['go_term_name'].nunique():,}")
print()

# Filter by score
df_comp_filtered = df_comp[
    df_comp["score"] >= COMPARTMENTS_SCORE_CUTOFF
].copy()

print(f"After score >= {COMPARTMENTS_SCORE_CUTOFF} filter:")
print(f"  Rows remaining    : {len(df_comp_filtered):,}")
print(f"  Unique proteins   : {df_comp_filtered['gene_symbol'].nunique():,}")
print()

# Build lookup: gene_symbol (upper) → set of GO compartment terms
compartment_lookup = {}

for _, row in df_comp_filtered.iterrows():
    gene = str(row["gene_symbol"]).upper()
    comp = str(row["go_term_name"])
    if gene not in compartment_lookup:
        compartment_lookup[gene] = set()
    compartment_lookup[gene].add(comp)

print(f"Compartment lookup built for : {len(compartment_lookup):,} proteins")
print()

# Check a few well-known RP proteins
test_proteins = ["RHO", "RPGR", "PRPF8", "CRX", "GNAT1"]
print("Sample compartments for known RP proteins:")
for p in test_proteins:
    comps = compartment_lookup.get(p, set())
    print(f"  {p}: {sorted(comps)[:5]}")

Loading COMPARTMENTS database...
Score cutoff: >= 3

Total rows in COMPARTMENTS file : 4,859,940
Unique proteins                 : 28,545
Unique compartments             : 3,312

After score >= 3 filter:
  Rows remaining    : 388,097
  Unique proteins   : 19,342

Compartment lookup built for : 19,342 proteins

Sample compartments for known RP proteins:
  RHO: ['9+0 non-motile cilium', 'Anchoring junction', 'Bounding membrane of organelle', 'Cell junction', 'Cell periphery']
  RPGR: ['9+0 non-motile cilium', '9+2 motile cilium', 'Axoneme', 'Cell projection', 'Cellular anatomical entity']
  PRPF8: ['Catalytic step 1 spliceosome', 'Catalytic step 2 spliceosome', 'Cellular anatomical entity', 'Cytoplasm', 'Intracellular anatomical structure']
  CRX: ['Cellular anatomical entity', 'Chromatin', 'Chromosome', 'GO:0005623', 'GO:0044424']
  GNAT1: ['9+0 non-motile cilium', 'Bounding membrane of organelle', 'Cell periphery', 'Cell projection', 'Cell projection membrane']


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 6 (corrected): Apply Filter A — COMPARTMENTS
#
# Problem identified:
#   Broad GO parent terms (e.g. "cellular_component",
#   "Cellular anatomical entity") are assigned to almost all
#   proteins, making the overlap check trivially true for every pair.
#   This means the filter was not discriminating.
#
# Solution:
#   Exclude overly broad/generic GO compartment terms and use
#   only specific subcellular compartments that are biologically
#   meaningful for physical co-localisation.
#
# Specific compartments retained:
#   Nucleus, Cytoplasm, Mitochondrion, Endoplasmic reticulum,
#   Golgi apparatus, Plasma membrane, Lysosome, Peroxisome,
#   Cytosol, Cell junction, Cilium, Photoreceptor outer segment,
#   Extracellular space, Nucleolus, Centrosome, Flagellum,
#   Axoneme, Synaptic vesicle, Spliceosome, etc.
#
# Broad terms EXCLUDED:
#   cellular_component, Cellular anatomical entity,
#   Intracellular anatomical structure, membrane-bounded organelle,
#   Organelle, Intracellular organelle, etc.
# ─────────────────────────────────────────────────────────────────

# Define broad GO terms to EXCLUDE from compartment matching
# These are parent/root terms that apply to nearly all proteins
BROAD_TERMS_TO_EXCLUDE = {
    "cellular_component",
    "cellular anatomical entity",
    "organelle",
    "intracellular anatomical structure",
    "membrane-bounded organelle",
    "intracellular organelle",
    "intracellular membrane-bounded organelle",
    "cell",
    "protein-containing complex",
    "cellular_component",
    "go:0005623",   # cell (GO ID form)
    "go:0044424",   # intracellular part
    "go:0005622",   # intracellular
    "go:0043226",   # organelle
    "go:0043229",   # intracellular organelle
    "go:0043227",   # membrane-bounded organelle
    "go:0043231",   # intracellular membrane-bounded organelle
    "go:0044464",   # cell part
}

# Rebuild compartment lookup using only specific terms
print("Rebuilding compartment lookup with specific terms only...")

compartment_lookup_specific = {}

for _, row in df_comp_filtered.iterrows():
    gene      = str(row["gene_symbol"]).upper()
    comp_name = str(row["go_term_name"]).lower()
    comp_id   = str(row["go_term_id"]).lower()
    comp_orig = str(row["go_term_name"])   # original case for display

    # Skip broad terms
    if comp_name in BROAD_TERMS_TO_EXCLUDE:
        continue
    if comp_id in BROAD_TERMS_TO_EXCLUDE:
        continue

    if gene not in compartment_lookup_specific:
        compartment_lookup_specific[gene] = set()
    compartment_lookup_specific[gene].add(comp_orig)

print(f"Proteins with specific compartment data : {len(compartment_lookup_specific):,}")
print()

# Verify with known RP proteins
print("Sample specific compartments for known RP proteins:")
test_proteins = ["RHO", "RPGR", "PRPF8", "CRX", "GNAT1"]
for p in test_proteins:
    comps = sorted(compartment_lookup_specific.get(p, set()))
    print(f"  {p}: {comps[:6]}")
print()

# Apply filter with specific compartments
print("Applying Filter A with specific compartments...")

results_A = []

for _, row in df_interactions.iterrows():
    pA = str(row["preferredName_A"]).upper()
    pB = str(row["preferredName_B"]).upper()

    comps_A = compartment_lookup_specific.get(pA, set())
    comps_B = compartment_lookup_specific.get(pB, set())

    if not comps_A and not comps_B:
        status = "both_unknown"
        overlap = set()
    elif not comps_A:
        status = "protein_A_unknown"
        overlap = set()
    elif not comps_B:
        status = "protein_B_unknown"
        overlap = set()
    else:
        overlap = comps_A & comps_B
        status = "co_localised" if overlap else "no_overlap"

    results_A.append({
        "preferredName_A"      : row["preferredName_A"],
        "preferredName_B"      : row["preferredName_B"],
        "score"                : row["score"],
        "protein_A_is_seed"    : row["protein_A_is_seed"],
        "protein_B_is_seed"    : row["protein_B_is_seed"],
        "compartment_status"   : status,
        "shared_compartments"  : "; ".join(sorted(overlap)) if overlap else "",
        "n_shared_compartments": len(overlap),
    })

df_filter_A = pd.DataFrame(results_A)

print("=" * 55)
print("FILTER A — COMPARTMENTS RESULTS (specific terms)")
print("=" * 55)
print(f"Total edges input    : {len(df_filter_A):,}")
print()
print("Compartment status breakdown:")
print(df_filter_A["compartment_status"].value_counts().to_string())
print()

# Edges that PASS
df_pass_A = df_filter_A[
    df_filter_A["compartment_status"] != "no_overlap"
].copy()
df_fail_A = df_filter_A[
    df_filter_A["compartment_status"] == "no_overlap"
].copy()

print(f"Edges PASSING Filter A : {len(df_pass_A):,}")
print(f"  co_localised         : {len(df_filter_A[df_filter_A['compartment_status']=='co_localised']):,}")
print(f"  unknown (kept)       : {len(df_filter_A[df_filter_A['compartment_status'].str.contains('unknown')]):,}")
print(f"Edges REMOVED          : {len(df_fail_A):,}")
print()

# Most common specific shared compartments
from collections import Counter
all_comps = []
for comps in df_filter_A["shared_compartments"]:
    if comps:
        all_comps.extend(comps.split("; "))
comp_counts = Counter(all_comps)

print("Top 15 most common SPECIFIC shared compartments:")
for comp, count in comp_counts.most_common(15):
    print(f"  {comp:<45} : {count:>5} edges")

# Save
out_A = "/content/rp_network/data/processed/filter_A_compartments.tsv"
df_filter_A.to_csv(out_A, sep="\t", index=False)
df_pass_A.to_csv(
    "/content/rp_network/data/processed/filter_A_passed.tsv",
    sep="\t", index=False
)
print()
print(f"✓ Full filter A table → filter_A_compartments.tsv")
print(f"✓ Passing edges       → filter_A_passed.tsv")

Rebuilding compartment lookup with specific terms only...
Proteins with specific compartment data : 18,807

Sample specific compartments for known RP proteins:
  RHO: ['9+0 non-motile cilium', 'Anchoring junction', 'Bounding membrane of organelle', 'Cell junction', 'Cell periphery', 'Cell projection']
  RPGR: ['9+0 non-motile cilium', '9+2 motile cilium', 'Axoneme', 'Cell projection', 'Centrosome', 'Ciliary basal body']
  PRPF8: ['Catalytic step 1 spliceosome', 'Catalytic step 2 spliceosome', 'Cytoplasm', 'Intracellular organelle lumen', 'Membrane', 'Nuclear body']
  CRX: ['Chromatin', 'Chromosome', 'Intracellular non-membrane-bounded organelle', 'Nucleus', 'non-membrane-bounded organelle']
  GNAT1: ['9+0 non-motile cilium', 'Bounding membrane of organelle', 'Cell periphery', 'Cell projection', 'Cell projection membrane', 'Ciliary membrane']

Applying Filter A with specific compartments...
FILTER A — COMPARTMENTS RESULTS (specific terms)
Total edges input    : 9,835

Compartment status

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 7 (corrected): Download Human Protein Atlas
#
# Database : Human Protein Atlas v25.1
# File     : rna_tissue_consensus.tsv
# URL      : https://www.proteinatlas.org/download/
# Access   : 2026-08-13
#
# HPA v25.1 provides RNA consensus tissue expression data for
# 20,162 human genes across 51 tissues including retina.
# Expression values are in nTPM (normalised transcripts per million).
#
# Note: The file URL changed between HPA versions.
# Current correct URL confirmed from the HPA download page v25.1.
# ─────────────────────────────────────────────────────────────────

HPA_URL  = "https://www.proteinatlas.org/download/tsv/rna_tissue_consensus.tsv.zip"
HPA_FILE = "/content/rp_network/data/raw/hpa_rna_consensus.tsv.zip"

print("Downloading Human Protein Atlas RNA consensus data (v25.1)...")
print(f"URL  : {HPA_URL}")
print("This may take 1-3 minutes...")
print()

r_hpa = requests.get(HPA_URL, timeout=300, stream=True)
print(f"Status : {r_hpa.status_code}")

if r_hpa.status_code == 200:
    with open(HPA_FILE, "wb") as f:
        for chunk in r_hpa.iter_content(chunk_size=8192):
            f.write(chunk)
    size_mb = Path(HPA_FILE).stat().st_size / 1_048_576
    print(f"✓ Downloaded: {size_mb:.1f} MB")
    print(f"  Saved to: {HPA_FILE}")
elif r_hpa.status_code == 404:
    # Try alternative URL format
    print("First URL failed. Trying alternative...")
    HPA_URL2 = "https://www.proteinatlas.org/download/rna_tissue_consensus.tsv.zip"
    r_hpa2 = requests.get(HPA_URL2, timeout=300, stream=True)
    print(f"Status : {r_hpa2.status_code}")
    if r_hpa2.status_code == 200:
        with open(HPA_FILE, "wb") as f:
            for chunk in r_hpa2.iter_content(chunk_size=8192):
                f.write(chunk)
        size_mb = Path(HPA_FILE).stat().st_size / 1_048_576
        print(f"✓ Downloaded: {size_mb:.1f} MB")
    else:
        print("✗ Both URLs failed.")
        print()
        print("Manual download instructions:")
        print("1. Go to https://www.proteinatlas.org/about/download")
        print("2. Under 'Tissue' section, find 'RNA expression (consensus)'")
        print("3. Click the download link for the TSV file")
        print("4. Upload the downloaded zip file to Colab")
        print("5. Place it at: /content/rp_network/data/raw/hpa_rna_consensus.tsv.zip")
else:
    print(f"✗ Error: {r_hpa.status_code}")

URL  : https://www.proteinatlas.org/download/tsv/rna_tissue_consensus.tsv.zip
This may take 1-3 minutes...

Status : 200
✓ Downloaded: 5.0 MB
  Saved to: /content/rp_network/data/raw/hpa_rna_consensus.tsv.zip


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 8: Parse HPA and extract retinal expression
# ─────────────────────────────────────────────────────────────────

import zipfile

print("Extracting and parsing HPA data...")

# Extract zip
with zipfile.ZipFile(HPA_FILE, "r") as z:
    names = z.namelist()
    print(f"Files in zip: {names}")
    z.extractall("/content/rp_network/data/raw/")

# Find extracted file
extracted = [f for f in names if f.endswith(".tsv")][0]
extracted_path = f"/content/rp_network/data/raw/{extracted}"

print(f"Extracted: {extracted_path}")
print()

# Load full file
df_hpa = pd.read_csv(extracted_path, sep="\t", low_memory=False)
print(f"HPA file columns : {list(df_hpa.columns)}")
print(f"Total rows       : {len(df_hpa):,}")
print(f"Unique tissues   : {df_hpa.iloc[:,2].nunique()}")
print()

# Identify correct column names
tissue_col = df_hpa.columns[2]   # usually "Tissue"
gene_col   = df_hpa.columns[1]   # usually "Gene name"
ntpm_col   = df_hpa.columns[3]   # usually "nTPM"

# Show available tissue names containing "retina"
retina_tissues = df_hpa[
    df_hpa[tissue_col].str.lower().str.contains("retina", na=False)
][tissue_col].unique()

print(f"Retina-related tissue names found: {retina_tissues}")

Extracting and parsing HPA data...
Files in zip: ['rna_tissue_consensus.tsv']
Extracted: /content/rp_network/data/raw/rna_tissue_consensus.tsv

HPA file columns : ['Gene', 'Gene name', 'Tissue', 'nTPM']
Total rows       : 1,026,292
Unique tissues   : 51

Retina-related tissue names found: ['retina']


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 9: Apply Filter B — Retinal Expression (HPA)
#
# Filter rule:
#   Both proteins in an interaction must have nTPM > 0 in retina.
#   nTPM = 0 → not detected → protein not expressed in retina
#   nTPM > 0 → expressed in retina → retained
#
# Applied to the edges that PASSED Filter A.
# ─────────────────────────────────────────────────────────────────

# Extract retinal expression only
RETINA_TISSUE = retina_tissues[0] if len(retina_tissues) > 0 else "retina"

df_retina = df_hpa[
    df_hpa[tissue_col].str.lower().str.contains("retina", na=False)
][[gene_col, ntpm_col]].copy()

df_retina.columns = ["gene_symbol", "ntpm_retina"]
df_retina["gene_symbol"] = df_retina["gene_symbol"].str.upper()
df_retina = df_retina.drop_duplicates(subset=["gene_symbol"])

# Build lookup: gene → nTPM in retina
retina_expr = dict(zip(df_retina["gene_symbol"], df_retina["ntpm_retina"]))

print(f"Retinal expression data loaded:")
print(f"  Proteins with retinal data : {len(retina_expr):,}")
print(f"  Expressed (nTPM > 0)       : {sum(1 for v in retina_expr.values() if v > 0):,}")
print(f"  Not expressed (nTPM = 0)   : {sum(1 for v in retina_expr.values() if v == 0):,}")
print()

# Apply to edges passing Filter A
results_B = []

for _, row in df_pass_A.iterrows():
    pA = str(row["preferredName_A"]).upper()
    pB = str(row["preferredName_B"]).upper()

    ntpm_A = retina_expr.get(pA, None)
    ntpm_B = retina_expr.get(pB, None)

    # Determine expression status
    if ntpm_A is None and ntpm_B is None:
        expr_status = "both_not_in_HPA"
    elif ntpm_A is None:
        expr_status = "protein_A_not_in_HPA"
    elif ntpm_B is None:
        expr_status = "protein_B_not_in_HPA"
    elif ntpm_A > 0 and ntpm_B > 0:
        expr_status = "both_expressed"
    elif ntpm_A == 0 and ntpm_B == 0:
        expr_status = "both_not_expressed"
    elif ntpm_A == 0:
        expr_status = "protein_A_not_expressed"
    else:
        expr_status = "protein_B_not_expressed"

    results_B.append({
        **row.to_dict(),
        "ntpm_retina_A"    : ntpm_A,
        "ntpm_retina_B"    : ntpm_B,
        "expression_status": expr_status,
    })

df_filter_B = pd.DataFrame(results_B)

print("=" * 55)
print("FILTER B — RETINAL EXPRESSION RESULTS")
print("=" * 55)
print(f"Input edges (from Filter A) : {len(df_filter_B)}")
print()
print("Expression status breakdown:")
print(df_filter_B["expression_status"].value_counts().to_string())
print()

# Edges that PASS Filter B
df_pass_B = df_filter_B[
    df_filter_B["expression_status"].isin([
        "both_expressed",
        "both_not_in_HPA",        # unknown → keep conservatively
        "protein_A_not_in_HPA",   # unknown → keep conservatively
        "protein_B_not_in_HPA",   # unknown → keep conservatively
    ])
].copy()

df_fail_B = df_filter_B[
    df_filter_B["expression_status"].isin([
        "both_not_expressed",
        "protein_A_not_expressed",
        "protein_B_not_expressed",
    ])
].copy()

print(f"Edges PASSING Filter B     : {len(df_pass_B)}")
print(f"  Both expressed in retina : {len(df_filter_B[df_filter_B['expression_status']=='both_expressed'])}")
print(f"  Unknown (kept)           : {len(df_pass_B) - len(df_filter_B[df_filter_B['expression_status']=='both_expressed'])}")
print(f"Edges REMOVED              : {len(df_fail_B)}")
print()

# Save
out_B = "/content/rp_network/data/processed/filter_B_expression.tsv"
df_filter_B.to_csv(out_B, sep="\t", index=False)
df_pass_B.to_csv(
    "/content/rp_network/data/processed/filter_B_passed.tsv",
    sep="\t", index=False
)
print(f"✓ Full filter B table → filter_B_expression.tsv")
print(f"✓ Passing edges       → filter_B_passed.tsv")

Retinal expression data loaded:
  Proteins with retinal data : 20,151
  Expressed (nTPM > 0)       : 16,460
  Not expressed (nTPM = 0)   : 3,691

FILTER B — RETINAL EXPRESSION RESULTS
Input edges (from Filter A) : 9312

Expression status breakdown:
expression_status
both_expressed             8740
protein_B_not_expressed     389
protein_B_not_in_HPA        183

Edges PASSING Filter B     : 8923
  Both expressed in retina : 8740
  Unknown (kept)           : 183
Edges REMOVED              : 389

✓ Full filter B table → filter_B_expression.tsv
✓ Passing edges       → filter_B_passed.tsv


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 10: Filtering summary so far
#
# Summary of what Filters A and B have done to the network.
# ─────────────────────────────────────────────────────────────────

print("=" * 55)
print("FILTERING PROGRESS SUMMARY")
print("=" * 55)
print()
print(f"Original network (Stage 4)  : {len(df_interactions):,} edges")
print()
print(f"After Filter A (COMPARTMENTS):")
print(f"  Edges removed (no overlap): {len(df_fail_A):,}")
print(f"  Edges remaining           : {len(df_pass_A):,}")
print()
print(f"After Filter B (HPA retinal expression):")
print(f"  Edges removed             : {len(df_fail_B):,}")
print(f"  Edges remaining           : {len(df_pass_B):,}")
print()

reduction_A = (1 - len(df_pass_A)/len(df_interactions)) * 100
reduction_AB = (1 - len(df_pass_B)/len(df_interactions)) * 100

print(f"Network reduction after Filter A    : {reduction_A:.1f}%")
print(f"Network reduction after Filters A+B : {reduction_AB:.1f}%")
print()

# Proteins remaining
proteins_remaining = (
    set(df_pass_B["preferredName_A"]) |
    set(df_pass_B["preferredName_B"])
)
seeds_remaining = {
    p for p in proteins_remaining
    if p.upper() in seed_upper
}
print(f"Proteins in filtered network : {len(proteins_remaining):,}")
print(f"  RP seed proteins retained  : {len(seeds_remaining)}")
print(f"  Partners retained          : {len(proteins_remaining) - len(seeds_remaining):,}")

FILTERING PROGRESS SUMMARY

Original network (Stage 4)  : 9,835 edges

After Filter A (COMPARTMENTS):
  Edges removed (no overlap): 523
  Edges remaining           : 9,312

After Filter B (HPA retinal expression):
  Edges removed             : 389
  Edges remaining           : 8,923

Network reduction after Filter A    : 5.3%
Network reduction after Filters A+B : 9.3%

Proteins in filtered network : 3,738
  RP seed proteins retained  : 105
  Partners retained          : 3,633


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 11: Save filtered network and update Drive
# ─────────────────────────────────────────────────────────────────

from google.colab import drive
import shutil
drive.mount("/content/drive")

# The filtered network after A + B is our working filtered network
# IID and TissueNet will add annotation scores but not hard-remove
# Save current filtered network as the primary output
df_pass_B.to_csv(
    "/content/rp_network/data/processed/network_filtered.tsv",
    sep="\t", index=False
)

# Save to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)

print("✓ Filtered network saved.")
print("✓ All files saved to Google Drive.")
print()
print("Files produced:")
print("  data/raw/compartments_human.tsv")
print("  data/raw/hpa_rna_consensus.tsv.zip")
print("  data/processed/filter_A_compartments.tsv")
print("  data/processed/filter_A_passed.tsv")
print("  data/processed/filter_B_expression.tsv")
print("  data/processed/filter_B_passed.tsv")
print("  data/processed/network_filtered.tsv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Filtered network saved.
✓ All files saved to Google Drive.

Files produced:
  data/raw/compartments_human.tsv
  data/raw/hpa_rna_consensus.tsv.zip
  data/processed/filter_A_compartments.tsv
  data/processed/filter_A_passed.tsv
  data/processed/filter_B_expression.tsv
  data/processed/filter_B_passed.tsv
  data/processed/network_filtered.tsv


In [ ]:
from google.colab import files
files.download("/content/rp_network/data/processed/network_filtered.tsv")
files.download("/content/rp_network/data/processed/filter_A_compartments.tsv")
files.download("/content/rp_network/data/processed/filter_B_expression.tsv")
files.download("/content/rp_network/docs/progress_log.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Drive if not already mounted
drive.mount("/content/drive")

# Copy everything from Colab to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)

print("✓ All files saved to Google Drive.")
print()
print("Files saved:")
drive_path = Path("/content/drive/MyDrive/rp_network")
for f in sorted(drive_path.rglob("*.csv")):
    size = f.stat().st_size / 1024
    print(f"  {f.relative_to(drive_path)} ({size:.1f} KB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!zip -r RP_Project1_Backup.zip .

Scanning files .
  adding: .config/ (stored 0%)
  adding: .config/.last_survey_prompt.yaml (stored 0%)
  adding: .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db (deflated 97%)
  adding: .config/gce (stored 0%)
  adding: .config/config_sentinel (stored 0%)
  adding: .config/.last_opt_in_prompt.yaml (stored 0%)
  adding: .config/configurations/ (stored 0%)
  adding: .config/configurations/config_default (deflated 15%)
  adding: .config/logs/ (stored 0%)
  adding: .config/logs/2026.08.10/ (stored 0%)
  adding: .config/logs/2026.08.10/13.25.25.328016.log (deflated 58%)
  adding: .config/logs/2026.08.10/13.26.03.888211.log (deflated 58%)
  adding: .config/logs/2026.08.10/13.26.19.834999.log (deflated 56%)
  adding: .config/logs/2026.08.10/13.26.18.998373.log (deflated 57%)
  adding: .config/logs/2026.08.10/13.25.43.233148.log (deflated 86%)
  adding: .config/logs/2026.08.10/13.24.58.373513.log (deflated 92%)
  adding: .config/default_configs.db (deflated 98%)
  adding

In [ ]:
from google.colab import files

files.download("RP_Project1_Backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Stage 5A — COMPARTMENTS annotation: supervisor-confirmed parameters

Following the supervisor meeting on 2026-08-28:

- STRING minimum combined score: ≥ 0.400
- COMPARTMENTS confidence score: ≥ 2.0
- HPA retinal expression criterion: nTPM > 0
- Context information will be retained as annotation rather than deleting
  interactions prematurely.
- Additional evidence from IID and TissueNet will subsequently be added to
  the same interaction-level evidence table.
- Database name, access date, parameters, and extraction method will be
  documented for reproducibility.

The previous exploratory COMPARTMENTS threshold of ≥3.0 is superseded by
the supervisor-confirmed threshold of ≥2.0.

In [ ]:
# =====================================================================
# NOTEBOOK 05 — STAGE 5A
# COMPARTMENTS INTERACTION ANNOTATION
#
# Supervisor-confirmed parameters:
#   STRING score           >= 0.400
#   COMPARTMENTS score     >= 2.0
#
# Confirmed: 2026-08-28
#
# IMPORTANT:
# This cell ANNOTATES all STRING interactions.
# It does NOT permanently delete UNKNOWN or NO_OVERLAP interactions.
# IID and TissueNet evidence will be added later before final filtering.
# =====================================================================

import pandas as pd
from pathlib import Path
from collections import defaultdict

# ---------------------------------------------------------------------
# 1. Project paths and confirmed parameters
# ---------------------------------------------------------------------

BASE = Path("/content/rp_network")

PPI_PATH = BASE / "data/processed/ppi_interactions.tsv"
COMP_PATH = BASE / "data/raw/compartments_human.tsv"

COMPARTMENTS_SCORE_CUTOFF = 2.0
COMPARTMENTS_ACCESS_DATE = "2026-08-28"

print("=" * 72)
print("STAGE 5A — COMPARTMENTS ANNOTATION")
print("=" * 72)
print(f"COMPARTMENTS confidence cutoff: >= {COMPARTMENTS_SCORE_CUTOFF}")
print(f"Database access/documentation date: {COMPARTMENTS_ACCESS_DATE}")


# ---------------------------------------------------------------------
# 2. Load original STRING interaction network
# ---------------------------------------------------------------------

ppi = pd.read_csv(
    PPI_PATH,
    sep="\t",
    low_memory=False
)

print(f"\nSTRING interactions loaded: {len(ppi):,}")

print(
    f"STRING score range: "
    f"{ppi['score'].min():.3f} – {ppi['score'].max():.3f}"
)

assert ppi["score"].min() >= 0.400, (
    "ERROR: Network contains STRING scores below 0.400."
)


# ---------------------------------------------------------------------
# 3. Normalize STRING protein identifiers
#
# STRING:       9606.ENSP00000295408
# COMPARTMENTS: ENSP00000295408
# ---------------------------------------------------------------------

ppi["protein_id_A"] = (
    ppi["stringId_A"]
    .astype(str)
    .str.replace(r"^9606\.", "", regex=True)
)

ppi["protein_id_B"] = (
    ppi["stringId_B"]
    .astype(str)
    .str.replace(r"^9606\.", "", regex=True)
)


# ---------------------------------------------------------------------
# 4. Load COMPARTMENTS human flat file
# ---------------------------------------------------------------------

comp = pd.read_csv(
    COMP_PATH,
    sep="\t",
    header=None,
    names=[
        "protein_id",
        "gene_symbol",
        "go_term_id",
        "go_term_name",
        "compartments_score",
        "evidence_source"
    ],
    low_memory=False
)

comp["protein_id"] = (
    comp["protein_id"]
    .astype(str)
    .str.strip()
)

comp["compartments_score"] = pd.to_numeric(
    comp["compartments_score"],
    errors="coerce"
)

print(f"COMPARTMENTS rows loaded: {len(comp):,}")


# ---------------------------------------------------------------------
# 5. Remove broad/non-informative GO cellular-component parent terms
#
# These broad ontology terms should not alone establish meaningful
# protein co-localisation.
# ---------------------------------------------------------------------

BROAD_GO_IDS = {
    "GO:0005575",  # cellular_component
    "GO:0110165",  # cellular anatomical entity
    "GO:0005623",  # cell
    "GO:0044464",  # cell part
    "GO:0005622",  # intracellular anatomical structure
    "GO:0043226",  # organelle
    "GO:0043227",  # membrane-bounded organelle
    "GO:0043229",  # intracellular organelle
    "GO:0043231",  # intracellular membrane-bounded organelle
    "GO:0016020",  # membrane
}

usable_comp = comp[
    (comp["compartments_score"] >= COMPARTMENTS_SCORE_CUTOFF) &
    (~comp["go_term_id"].isin(BROAD_GO_IDS))
].copy()

print(
    f"Usable COMPARTMENTS rows at score >= "
    f"{COMPARTMENTS_SCORE_CUTOFF}: {len(usable_comp):,}"
)


# ---------------------------------------------------------------------
# 6. Build protein ID -> GO localisation lookup
#
# For duplicated protein/GO combinations, retain the highest score.
# ---------------------------------------------------------------------

go_lookup = defaultdict(dict)

for row in usable_comp.itertuples(index=False):

    pid = row.protein_id
    go_id = row.go_term_id

    current = go_lookup[pid].get(go_id)

    if (
        current is None
        or row.compartments_score > current["score"]
    ):
        go_lookup[pid][go_id] = {
            "name": row.go_term_name,
            "score": row.compartments_score
        }

print(
    "Proteins with usable COMPARTMENTS annotations:",
    f"{len(go_lookup):,}"
)


# ---------------------------------------------------------------------
# 7. Annotate every STRING interaction
# ---------------------------------------------------------------------

records = []

for row in ppi.itertuples(index=False):

    terms_A = go_lookup.get(row.protein_id_A, {})
    terms_B = go_lookup.get(row.protein_id_B, {})

    ids_A = set(terms_A.keys())
    ids_B = set(terms_B.keys())

    shared_ids = ids_A & ids_B

    # Classification
    if not ids_A or not ids_B:
        compartment_status = "UNKNOWN"

    elif shared_ids:
        compartment_status = "PASS"

    else:
        compartment_status = "NO_OVERLAP"

    # Human-readable annotations
    names_A = sorted({
        item["name"]
        for item in terms_A.values()
    })

    names_B = sorted({
        item["name"]
        for item in terms_B.values()
    })

    shared_names = sorted({
        terms_A[go_id]["name"]
        for go_id in shared_ids
    })

    # Shared-term confidence:
    # use the weaker score of the two proteins for each shared GO term,
    # then report the maximum among shared terms.
    shared_scores = []

    for go_id in shared_ids:
        shared_scores.append(
            min(
                terms_A[go_id]["score"],
                terms_B[go_id]["score"]
            )
        )

    max_shared_confidence = (
        max(shared_scores)
        if shared_scores
        else None
    )

    records.append({

        # Original interaction identity
        "protein_A": row.preferredName_A,
        "protein_B": row.preferredName_B,
        "protein_id_A": row.protein_id_A,
        "protein_id_B": row.protein_id_B,

        # STRING evidence
        "string_score": row.score,
        "string_escore": row.escore,
        "string_dscore": row.dscore,
        "string_tscore": row.tscore,

        # Seed status
        "A_is_RP_seed": row.protein_A_is_seed,
        "B_is_RP_seed": row.protein_B_is_seed,

        # COMPARTMENTS annotation
        "A_n_specific_compartments": len(ids_A),
        "B_n_specific_compartments": len(ids_B),

        "A_compartments": "; ".join(names_A),
        "B_compartments": "; ".join(names_B),

        "n_shared_compartments": len(shared_ids),
        "shared_GO_IDs": "; ".join(sorted(shared_ids)),
        "shared_compartments": "; ".join(shared_names),

        "max_shared_compartment_score": max_shared_confidence,

        "compartment_status": compartment_status,

        # Reproducibility metadata
        "compartments_database": "COMPARTMENTS",
        "compartments_score_cutoff": COMPARTMENTS_SCORE_CUTOFF,
        "compartments_access_date": COMPARTMENTS_ACCESS_DATE
    })


comp_t2 = pd.DataFrame(records)


# ---------------------------------------------------------------------
# 8. Verify that no STRING interactions were lost
# ---------------------------------------------------------------------

assert len(comp_t2) == len(ppi), (
    "ERROR: Number of interactions changed during annotation."
)

print("\n" + "=" * 72)
print("COMPARTMENTS RESULTS")
print("=" * 72)

print(f"Original STRING interactions : {len(ppi):,}")
print(f"Annotated interactions       : {len(comp_t2):,}")

print("\nStatus counts:")
print(
    comp_t2["compartment_status"]
    .value_counts()
)

print("\nStatus percentages:")
print(
    (
        comp_t2["compartment_status"]
        .value_counts(normalize=True) * 100
    ).round(2)
)


# ---------------------------------------------------------------------
# 9. Display representative examples
# ---------------------------------------------------------------------

print("\nRepresentative examples:")

for status in ["PASS", "NO_OVERLAP", "UNKNOWN"]:

    subset = comp_t2[
        comp_t2["compartment_status"] == status
    ]

    print("\n" + "-" * 72)
    print(f"{status}: {len(subset):,}")
    print("-" * 72)

    if len(subset) > 0:

        display(
            subset[
                [
                    "protein_A",
                    "protein_B",
                    "string_score",
                    "A_n_specific_compartments",
                    "B_n_specific_compartments",
                    "n_shared_compartments",
                    "shared_compartments",
                    "max_shared_compartment_score",
                    "compartment_status"
                ]
            ].head(10)
        )


# ---------------------------------------------------------------------
# 10. Save complete annotation table
#
# We use a dedicated threshold-2 file so the old threshold-3
# exploratory result remains available for provenance.
# ---------------------------------------------------------------------

OUTPUT_ALL = (
    BASE /
    "data/processed/"
    "compartments_interaction_annotation_t2.tsv"
)

comp_t2.to_csv(
    OUTPUT_ALL,
    sep="\t",
    index=False
)


# ---------------------------------------------------------------------
# 11. Save convenience subset with shared localisation
#
# IMPORTANT:
# This is NOT the final context-specific network.
# HPA, IID, and TissueNet evidence still need to be integrated.
# ---------------------------------------------------------------------

OUTPUT_SHARED = (
    BASE /
    "data/processed/"
    "compartments_shared_t2.tsv"
)

comp_t2[
    comp_t2["compartment_status"] == "PASS"
].to_csv(
    OUTPUT_SHARED,
    sep="\t",
    index=False
)


# ---------------------------------------------------------------------
# 12. Final save summary
# ---------------------------------------------------------------------

print("\nSaved:")
print(f"1. {OUTPUT_ALL}")
print(f"2. {OUTPUT_SHARED}")

print("\nIMPORTANT:")
print(
    "The full annotation file retains PASS, NO_OVERLAP, and UNKNOWN "
    "interactions."
)
print(
    "Do not treat compartments_shared_t2.tsv as the final network yet."
)
print(
    "HPA, IID, and TissueNet context evidence must still be integrated."
)

STAGE 5A — COMPARTMENTS ANNOTATION
COMPARTMENTS confidence cutoff: >= 2.0
Database access/documentation date: 2026-08-28

STRING interactions loaded: 9,835
STRING score range: 0.400 – 0.999
COMPARTMENTS rows loaded: 4,859,940
Usable COMPARTMENTS rows at score >= 2.0: 526,771
Proteins with usable COMPARTMENTS annotations: 20,097

COMPARTMENTS RESULTS
Original STRING interactions : 9,835
Annotated interactions       : 9,835

Status counts:
compartment_status
PASS          9751
NO_OVERLAP      71
UNKNOWN         13
Name: count, dtype: int64

Status percentages:
compartment_status
PASS          99.15
NO_OVERLAP     0.72
UNKNOWN        0.13
Name: proportion, dtype: float64

Representative examples:

------------------------------------------------------------------------
PASS: 9,751
------------------------------------------------------------------------


,protein_A,protein_B,string_score,A_n_specific_compartments,B_n_specific_compartments,n_shared_compartments,shared_compartments,max_shared_compartment_score,compartment_status
0,MERTK,PROS1,0.999,76,48,23,Cell periphery; Complement component C1q compl...,5.000,PASS
1,MERTK,GAS6,0.999,76,75,55,Anchoring junction; Apoptotic body; Cell junct...,5.000,PASS
2,MVK,HMGCR,0.999,21,120,18,Cell periphery; Chloroplast; Cytoplasm; Cytoso...,5.000,PASS
3,MVK,MVD,0.999,21,11,10,Cytoplasm; Cytosol; Dxr protein complex; Extra...,5.000,PASS
4,DYNC2H1,WDR34,0.999,63,35,31,Axonemal dynein complex; Axoneme; BBSome; Cell...,5.000,PASS
5,CNGB1,CNGA4,0.999,48,35,30,Bounding membrane of organelle; Cation channel...,4.631,PASS
6,CNGB1,CNGA1,0.999,48,29,24,9+0 non-motile cilium; Bounding membrane of or...,4.478,PASS
7,CNGB1,CNGA2,0.999,48,39,32,Bounding membrane of organelle; Cation channel...,4.648,PASS
8,PRPF8,LSM8,0.999,38,27,22,Cytoplasm; Cytosol; Intracellular organelle lu...,5.000,PASS
9,PRPF8,CDC40,0.999,38,24,23,Catalytic step 1 spliceosome; Catalytic step 2...,5.000,PASS



------------------------------------------------------------------------
NO_OVERLAP: 71
------------------------------------------------------------------------


,protein_A,protein_B,string_score,A_n_specific_compartments,B_n_specific_compartments,n_shared_compartments,shared_compartments,max_shared_compartment_score,compartment_status
1159,COQ2,SALL3,0.901,42,1,0,,NaN,NO_OVERLAP
1813,IMPG2,ZNF513,0.807,13,5,0,,NaN,NO_OVERLAP
2080,EYS,ZNF513,0.774,41,5,0,,NaN,NO_OVERLAP
2432,RDH12,ZNF513,0.738,20,5,0,,NaN,NO_OVERLAP
2442,RAX2,CDHR1,0.737,6,21,0,,NaN,NO_OVERLAP
2466,IMPG2,IDH3B,0.733,13,22,0,,NaN,NO_OVERLAP
2519,ZNF408,NDP,0.728,8,8,0,,NaN,NO_OVERLAP
2565,IMPG2,RP9,0.724,13,25,0,,NaN,NO_OVERLAP
2584,PRPF3,IMPG2,0.722,34,13,0,,NaN,NO_OVERLAP
2587,DHX38,LRTOMT-2,0.722,28,11,0,,NaN,NO_OVERLAP



------------------------------------------------------------------------
UNKNOWN: 13
------------------------------------------------------------------------


,protein_A,protein_B,string_score,A_n_specific_compartments,B_n_specific_compartments,n_shared_compartments,shared_compartments,max_shared_compartment_score,compartment_status
1430,ARHGEF18,ENSP00000500339,0.861,31,0,0,,NaN,UNKNOWN
3161,OFD1,CXorf51A,0.670,57,0,0,,NaN,UNKNOWN
4732,IDH3B,ENSP00000497305,0.563,22,0,0,,NaN,UNKNOWN
4996,ARL3,ENSP00000387111,0.549,71,0,0,,NaN,UNKNOWN
5941,COQ2,ENSP00000497305,0.510,42,0,0,,NaN,UNKNOWN
6239,DHDDS,H0YIJ7_HUMAN,0.500,19,0,0,,NaN,UNKNOWN
6458,IFT172,CXorf51A,0.491,64,0,0,,NaN,UNKNOWN
7738,PRPF6,TMEM236,0.448,34,0,0,,NaN,UNKNOWN
8915,BBS2,ENSP00000398526,0.419,54,0,0,,NaN,UNKNOWN
8926,KIZ,ENSP00000470615,0.419,14,0,0,,NaN,UNKNOWN



Saved:
1. /content/rp_network/data/processed/compartments_interaction_annotation_t2.tsv
2. /content/rp_network/data/processed/compartments_shared_t2.tsv

IMPORTANT:
The full annotation file retains PASS, NO_OVERLAP, and UNKNOWN interactions.
Do not treat compartments_shared_t2.tsv as the final network yet.
HPA, IID, and TissueNet context evidence must still be integrated.


## Stage 5B — Integration of retinal expression evidence

Retinal RNA expression evidence from the Human Protein Atlas (HPA)
was integrated with the COMPARTMENTS-annotated STRING interaction network.

Supervisor-confirmed criterion:

- Tissue: retina
- Expression measure: nTPM
- Retinal expression criterion: nTPM > 0

All interactions are retained during evidence integration. Interactions are
annotated according to whether both interacting proteins have detectable
retinal expression. Missing HPA information is classified as UNKNOWN rather
than interpreted as absence of expression.

This combined evidence table will subsequently be extended with IID and
TissueNet evidence before final context-specific network selection.

In [ ]:
# =====================================================================
# NOTEBOOK 05 — STAGE 5B
# INTEGRATE COMPARTMENTS >=2 WITH HPA RETINA EXPRESSION >0
#
# Supervisor-confirmed parameters:
#   STRING        >= 0.400
#   COMPARTMENTS  >= 2.0
#   HPA retina    nTPM > 0
#
# IMPORTANT:
# This creates an integrated EVIDENCE TABLE.
# It is NOT yet the final filtered network.
# IID and TissueNet will be added afterwards.
# =====================================================================

import pandas as pd
from pathlib import Path

BASE = Path("/content/rp_network")

COMP_FILE = (
    BASE /
    "data/processed/compartments_interaction_annotation_t2.tsv"
)

HPA_FILE = (
    BASE /
    "data/processed/hpa_retina_interaction_annotation.tsv"
)

OUTPUT_FILE = (
    BASE /
    "data/processed/context_specific_evidence_master.tsv"
)

print("=" * 72)
print("STAGE 5B — COMPARTMENTS + HPA RETINA INTEGRATION")
print("=" * 72)


# ---------------------------------------------------------------------
# 1. Load COMPARTMENTS annotation
# ---------------------------------------------------------------------

comp = pd.read_csv(
    COMP_FILE,
    sep="\t",
    low_memory=False
)

print(f"\nCOMPARTMENTS interactions : {len(comp):,}")


# ---------------------------------------------------------------------
# 2. Load HPA retina annotation
# ---------------------------------------------------------------------

hpa = pd.read_csv(
    HPA_FILE,
    sep="\t",
    low_memory=False
)

print(f"HPA interactions          : {len(hpa):,}")


# ---------------------------------------------------------------------
# 3. Basic checks
# ---------------------------------------------------------------------

assert len(comp) == 9835, (
    f"Expected 9,835 COMPARTMENTS rows, found {len(comp):,}"
)

assert len(hpa) == 9835, (
    f"Expected 9,835 HPA rows, found {len(hpa):,}"
)


# ---------------------------------------------------------------------
# 4. Check required HPA columns
# ---------------------------------------------------------------------

required_hpa_columns = [
    "protein_id_A",
    "protein_id_B",
    "retina_nTPM_A",
    "retina_nTPM_B",
    "A_in_HPA_retina",
    "B_in_HPA_retina",
    "expression_status_gt0"
]

missing = [
    col for col in required_hpa_columns
    if col not in hpa.columns
]

if missing:
    raise ValueError(
        f"Missing required HPA columns: {missing}"
    )

print("\nRequired HPA columns found.")


# ---------------------------------------------------------------------
# 5. Verify interaction identifiers
# ---------------------------------------------------------------------

comp_keys = set(
    zip(
        comp["protein_id_A"],
        comp["protein_id_B"]
    )
)

hpa_keys = set(
    zip(
        hpa["protein_id_A"],
        hpa["protein_id_B"]
    )
)

print(
    f"Interaction pairs identical: "
    f"{comp_keys == hpa_keys}"
)

assert comp_keys == hpa_keys, (
    "COMPARTMENTS and HPA interaction pairs do not match."
)


# ---------------------------------------------------------------------
# 6. Keep only HPA columns needed for integration
# ---------------------------------------------------------------------

hpa_context = hpa[
    required_hpa_columns
].copy()


# ---------------------------------------------------------------------
# 7. Merge COMPARTMENTS + HPA
# ---------------------------------------------------------------------

context = comp.merge(
    hpa_context,
    on=[
        "protein_id_A",
        "protein_id_B"
    ],
    how="left",
    validate="one_to_one"
)

assert len(context) == len(comp)

print(
    f"\nIntegrated interactions   : {len(context):,}"
)


# ---------------------------------------------------------------------
# 8. Add simple evidence flags
# ---------------------------------------------------------------------

context["compartment_supported"] = (
    context["compartment_status"] == "PASS"
)

context["both_retina_expressed"] = (
    context["expression_status_gt0"] == "BOTH_EXPRESSED"
)


# ---------------------------------------------------------------------
# 9. Create current combined evidence status
#
# This is NOT the final context-specific decision.
# ---------------------------------------------------------------------

def classify_current_context(row):

    if (
        row["compartment_status"] == "UNKNOWN"
        or row["expression_status_gt0"] == "UNKNOWN"
    ):
        return "UNKNOWN"

    if (
        row["compartment_status"] == "PASS"
        and
        row["expression_status_gt0"] == "BOTH_EXPRESSED"
    ):
        return "SUPPORTED"

    return "CONFLICT"


context["current_context_status"] = context.apply(
    classify_current_context,
    axis=1
)


# ---------------------------------------------------------------------
# 10. Add reproducibility metadata
# ---------------------------------------------------------------------

context["string_threshold"] = 0.400

context["compartments_threshold"] = 2.0
context["compartments_database"] = "COMPARTMENTS"

context["expression_database"] = "Human Protein Atlas"
context["expression_tissue"] = "retina"
context["expression_measure"] = "nTPM"
context["expression_threshold"] = ">0"


# ---------------------------------------------------------------------
# 11. Print summary
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("CURRENT CONTEXT EVIDENCE SUMMARY")
print("=" * 72)

print(
    context["current_context_status"]
    .value_counts()
)

print("\nPercentages:")

print(
    (
        context["current_context_status"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)


# ---------------------------------------------------------------------
# 12. COMPARTMENTS × HPA cross-tabulation
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("COMPARTMENTS × HPA")
print("=" * 72)

cross_tab = pd.crosstab(
    context["compartment_status"],
    context["expression_status_gt0"],
    margins=True
)

display(cross_tab)


# ---------------------------------------------------------------------
# 13. Show representative examples
# ---------------------------------------------------------------------

for status in ["SUPPORTED", "CONFLICT", "UNKNOWN"]:

    subset = context[
        context["current_context_status"] == status
    ]

    print("\n" + "-" * 72)
    print(f"{status}: {len(subset):,}")
    print("-" * 72)

    if len(subset) > 0:

        display(
            subset[
                [
                    "protein_A",
                    "protein_B",
                    "string_score",
                    "compartment_status",
                    "n_shared_compartments",
                    "shared_compartments",
                    "retina_nTPM_A",
                    "retina_nTPM_B",
                    "expression_status_gt0",
                    "current_context_status"
                ]
            ].head(10)
        )


# ---------------------------------------------------------------------
# 14. Save MASTER evidence table
#
# IID and TissueNet columns will later be added to THIS file.
# ---------------------------------------------------------------------

context.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False
)

print("\n" + "=" * 72)
print("SAVED")
print("=" * 72)

print(OUTPUT_FILE)

print("\nIMPORTANT:")
print("This is the current context-specific EVIDENCE table.")
print("It is NOT yet the final filtered network.")
print("Next evidence sources: IID and TissueNet.")

STAGE 5B — COMPARTMENTS + HPA RETINA INTEGRATION

COMPARTMENTS interactions : 9,835
HPA interactions          : 9,835

Required HPA columns found.
Interaction pairs identical: True

Integrated interactions   : 9,835

CURRENT CONTEXT EVIDENCE SUMMARY
current_context_status
SUPPORTED    9149
CONFLICT      485
UNKNOWN       201
Name: count, dtype: int64

Percentages:
current_context_status
SUPPORTED    93.02
CONFLICT      4.93
UNKNOWN       2.04
Name: proportion, dtype: float64

COMPARTMENTS × HPA


expression_status_gt0,BELOW_THRESHOLD,BOTH_EXPRESSED,UNKNOWN,All
compartment_status,,,,
NO_OVERLAP,17,52,2,71
PASS,416,9149,186,9751
UNKNOWN,4,1,8,13
All,437,9202,196,9835



------------------------------------------------------------------------
SUPPORTED: 9,149
------------------------------------------------------------------------


,protein_A,protein_B,string_score,compartment_status,n_shared_compartments,shared_compartments,retina_nTPM_A,retina_nTPM_B,expression_status_gt0,current_context_status
0,MERTK,PROS1,0.999,PASS,23,Cell periphery; Complement component C1q compl...,15.4,4.3,BOTH_EXPRESSED,SUPPORTED
1,MERTK,GAS6,0.999,PASS,55,Anchoring junction; Apoptotic body; Cell junct...,15.4,5.8,BOTH_EXPRESSED,SUPPORTED
2,MVK,HMGCR,0.999,PASS,18,Cell periphery; Chloroplast; Cytoplasm; Cytoso...,3.2,17.4,BOTH_EXPRESSED,SUPPORTED
3,MVK,MVD,0.999,PASS,10,Cytoplasm; Cytosol; Dxr protein complex; Extra...,3.2,3.2,BOTH_EXPRESSED,SUPPORTED
5,CNGB1,CNGA4,0.999,PASS,30,Bounding membrane of organelle; Cation channel...,557.0,1.0,BOTH_EXPRESSED,SUPPORTED
6,CNGB1,CNGA1,0.999,PASS,24,9+0 non-motile cilium; Bounding membrane of or...,557.0,244.8,BOTH_EXPRESSED,SUPPORTED
8,PRPF8,LSM8,0.999,PASS,22,Cytoplasm; Cytosol; Intracellular organelle lu...,76.4,15.4,BOTH_EXPRESSED,SUPPORTED
9,PRPF8,CDC40,0.999,PASS,23,Catalytic step 1 spliceosome; Catalytic step 2...,76.4,13.7,BOTH_EXPRESSED,SUPPORTED
10,PRPF8,DDX23,0.999,PASS,29,Catalytic step 1 spliceosome; Catalytic step 2...,76.4,17.9,BOTH_EXPRESSED,SUPPORTED
11,PRPF8,EFTUD2,0.999,PASS,33,Catalytic step 1 spliceosome; Catalytic step 2...,76.4,14.4,BOTH_EXPRESSED,SUPPORTED



------------------------------------------------------------------------
CONFLICT: 485
------------------------------------------------------------------------


,protein_A,protein_B,string_score,compartment_status,n_shared_compartments,shared_compartments,retina_nTPM_A,retina_nTPM_B,expression_status_gt0,current_context_status
7,CNGB1,CNGA2,0.999,PASS,32,Bounding membrane of organelle; Cation channel...,557.0,0.0,BELOW_THRESHOLD,CONFLICT
656,IMPDH1,AMPD1,0.963,PASS,16,Cytoplasm; Cytosol; GMP reductase complex; GO:...,98.6,0.0,BELOW_THRESHOLD,CONFLICT
664,CNGB1,CALML5,0.961,PASS,34,Axon; Axon terminus; Bounding membrane of orga...,557.0,0.0,BELOW_THRESHOLD,CONFLICT
669,CNGB1,CALML3,0.961,PASS,34,Axon; Axon terminus; Bounding membrane of orga...,557.0,0.0,BELOW_THRESHOLD,CONFLICT
684,CREB3,DCSTAMP,0.959,PASS,14,Bounding membrane of organelle; Cytoplasm; End...,21.7,0.0,BELOW_THRESHOLD,CONFLICT
690,CREB3,CREB3L3,0.958,PASS,26,Bounding membrane of organelle; Chromatin; Chr...,21.7,0.0,BELOW_THRESHOLD,CONFLICT
691,MVK,IGHD,0.958,PASS,4,Cell periphery; Extracellular region; Plasma m...,3.2,0.0,BELOW_THRESHOLD,CONFLICT
761,CREB3,CREB3L1,0.947,PASS,19,Chromatin; Chromosome; Cytoplasm; Cytosol; End...,21.7,0.0,BELOW_THRESHOLD,CONFLICT
861,HK1,FBP2,0.932,PASS,9,6-phosphofructokinase complex; Cytoplasm; Cyto...,141.7,0.0,BELOW_THRESHOLD,CONFLICT
862,LRAT,ADH1A,0.932,PASS,9,Cell periphery; Cytoplasm; Intracellular non-m...,0.9,0.0,BELOW_THRESHOLD,CONFLICT



------------------------------------------------------------------------
UNKNOWN: 201
------------------------------------------------------------------------


,protein_A,protein_B,string_score,compartment_status,n_shared_compartments,shared_compartments,retina_nTPM_A,retina_nTPM_B,expression_status_gt0,current_context_status
4,DYNC2H1,WDR34,0.999,PASS,31,Axonemal dynein complex; Axoneme; BBSome; Cell...,9.5,NaN,UNKNOWN,UNKNOWN
63,DYNC2H1,WDR60,0.999,PASS,31,Axonemal dynein complex; Axoneme; BBSome; Cell...,9.5,NaN,UNKNOWN,UNKNOWN
224,CRB1,MPP5,0.999,PASS,20,Adherens junction; Anchoring junction; Apical ...,69.2,NaN,UNKNOWN,UNKNOWN
516,GUCY2D,GUCA1A-2,0.985,PASS,12,9+0 non-motile cilium; Cell projection; Cilium...,31.7,NaN,UNKNOWN,UNKNOWN
654,DYNC2H1,TCTEX1D2,0.964,PASS,25,Axonemal dynein complex; Axoneme; Cell project...,9.5,NaN,UNKNOWN,UNKNOWN
756,SNRNP200,M0R3G1_HUMAN,0.948,PASS,11,Nucleus; Ribonucleoprotein complex; Sm-like pr...,62.7,NaN,UNKNOWN,UNKNOWN
804,HK1,G6PC,0.941,PASS,30,6-phosphofructokinase complex; Cell periphery;...,141.7,NaN,UNKNOWN,UNKNOWN
900,HK1,AMDHD2-2,0.926,PASS,2,Integral component of membrane; Intrinsic comp...,141.7,NaN,UNKNOWN,UNKNOWN
922,PRPF8,M0R3G1_HUMAN,0.923,PASS,11,Nucleus; Ribonucleoprotein complex; Sm-like pr...,76.4,NaN,UNKNOWN,UNKNOWN
986,OFD1,FOPNL,0.916,PASS,15,Cell projection; Centriolar satellite; Centrio...,2.3,NaN,UNKNOWN,UNKNOWN



SAVED
/content/rp_network/data/processed/context_specific_evidence_master.tsv

IMPORTANT:
This is the current context-specific EVIDENCE table.
It is NOT yet the final filtered network.
Next evidence sources: IID and TissueNet.


## Stage 5C — Integrated Interactions Database (IID)

**Database:** Integrated Interactions Database (IID)  
**Access date:** 2026-08-28  
**Purpose:** Add independent interaction and tissue/context evidence to the
STRING-derived RP interaction network.

IID integrates experimentally detected, orthology-supported, and predicted
protein-protein interactions and annotates PPIs with biological context,
including tissues and subcellular localization.

Accepted query identifiers include:
- Gene symbols
- UniProt identifiers
- Entrez Gene identifiers

For this project, IID will be evaluated as an additional evidence source
rather than replacing the STRING network. Relevant IID information will be
added as annotation columns to the existing context-specific evidence table.

In [ ]:
# =====================================================================
# STAGE 5C — PREPARE STRING INTERACTIONS FOR IID
# =====================================================================

import pandas as pd
from pathlib import Path

BASE = Path("/content/rp_network")

PPI_PATH = BASE / "data/processed/ppi_interactions.tsv"
OUTPUT = BASE / "data/processed/iid_query_pairs.txt"

ppi = pd.read_csv(PPI_PATH, sep="\t", low_memory=False)

print("STRING interactions:", len(ppi))
print("\nColumns:")
print(ppi.columns.tolist())

# Extract the two gene symbols for every STRING interaction
pairs = ppi[
    ["preferredName_A", "preferredName_B"]
].copy()

# Remove missing identifiers
pairs = pairs.dropna()

# Strip whitespace
pairs["preferredName_A"] = pairs["preferredName_A"].astype(str).str.strip()
pairs["preferredName_B"] = pairs["preferredName_B"].astype(str).str.strip()

# Write each interaction as two consecutive gene symbols.
# IID interprets consecutive pairs as interactions.
with open(OUTPUT, "w") as f:
    for row in pairs.itertuples(index=False):
        f.write(f"{row.preferredName_A}\t{row.preferredName_B}\n")

print("\nIID interaction query file created:")
print(OUTPUT)

print("\nNumber of interaction pairs:", len(pairs))

print("\nFirst 10:")
display(pairs.head(10))

STRING interactions: 9835

Columns:
['stringId_A', 'stringId_B', 'preferredName_A', 'preferredName_B', 'ncbiTaxonId', 'score', 'nscore', 'fscore', 'pscore', 'ascore', 'escore', 'dscore', 'tscore', 'query_gene', 'query_is_seed', 'protein_A_is_seed', 'protein_B_is_seed']

IID interaction query file created:
/content/rp_network/data/processed/iid_query_pairs.txt

Number of interaction pairs: 9835

First 10:


,preferredName_A,preferredName_B
0,MERTK,PROS1
1,MERTK,GAS6
2,MVK,HMGCR
3,MVK,MVD
4,DYNC2H1,WDR34
5,CNGB1,CNGA4
6,CNGB1,CNGA1
7,CNGB1,CNGA2
8,PRPF8,LSM8
9,PRPF8,CDC40


In [ ]:
from google.colab import files

files.download(
    "/content/rp_network/data/processed/iid_query_pairs.txt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import gzip

iid_path = "/content/rp_network/data/raw/human_annotated_PPIs.txt.gz"

print("Testing ENTIRE gzip file...")

try:
    with gzip.open(iid_path, "rb") as f:
        while f.read(1024 * 1024):
            pass

    print("✓ FULL GZIP FILE IS VALID")

except Exception as e:
    print("✗ FILE IS STILL CORRUPTED")
    print(type(e).__name__, e)

Testing ENTIRE gzip file...
✓ FULL GZIP FILE IS VALID


In [ ]:
from pathlib import Path

p = Path("/content/rp_network/data/raw/human_annotated_PPIs.txt.gz")

print("Bytes:", p.stat().st_size)
print("MB:", round(p.stat().st_size / (1024**2), 2))

Bytes: 150992743
MB: 144.0


In [ ]:
# =====================================================================
# NOTEBOOK 05 — STAGE 5C
# IID (INTEGRATED INTERACTIONS DATABASE) ANNOTATION
#
# PURPOSE
# -------
# Match the 9,835 STRING interactions against the complete human IID
# interaction dataset and add IID evidence to every STRING edge.
#
# IMPORTANT
# ---------
# - All original STRING interactions are retained.
# - Matching is UNDIRECTED:
#       MERTK--GAS6 == GAS6--MERTK
# - IID absence means "no IID evidence found", NOT "false interaction".
# - IID is an additional evidence layer.
# =====================================================================

import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# 1. Paths and metadata
# ---------------------------------------------------------------------

BASE = Path("/content/rp_network")

STRING_FILE = BASE / "data/processed/ppi_interactions.tsv"
IID_FILE = BASE / "data/raw/human_annotated_PPIs.txt.gz"

OUTPUT_FILE = BASE / "data/processed/iid_interaction_annotation.tsv"

IID_DATABASE = "Integrated Interactions Database (IID)"
IID_ACCESS_DATE = "2026-08-28"

print("=" * 72)
print("STAGE 5C — IID INTERACTION ANNOTATION")
print("=" * 72)

assert STRING_FILE.exists(), f"STRING file not found: {STRING_FILE}"
assert IID_FILE.exists(), f"IID file not found: {IID_FILE}"

print("\nSTRING file:")
print(STRING_FILE)

print("\nIID file:")
print(IID_FILE)


# ---------------------------------------------------------------------
# 2. Load STRING interaction table
# ---------------------------------------------------------------------

string = pd.read_csv(
    STRING_FILE,
    sep="\t",
    low_memory=False
)

print("\nSTRING interactions loaded:", f"{len(string):,}")

assert len(string) == 9835, (
    f"Expected 9,835 STRING interactions, found {len(string):,}"
)

required_string_cols = [
    "preferredName_A",
    "preferredName_B",
    "score"
]

for col in required_string_cols:
    assert col in string.columns, f"Missing STRING column: {col}"


# ---------------------------------------------------------------------
# 3. Load the IID columns needed for interaction evidence
# ---------------------------------------------------------------------

iid_columns = [
    "uniprot1",
    "uniprot2",
    "symbol1",
    "symbol2",
    "methods",
    "pmids",
    "db_with_ppi",
    "evidence_type",
    "n_exp_methods",
    "n_exp_pmids",
    "n_pred_pmids",
    "n_pmids",
    "detection_type"
]

print("\nLoading IID dataset...")

iid = pd.read_csv(
    IID_FILE,
    sep="\t",
    compression="gzip",
    usecols=iid_columns,
    low_memory=False
)

print("IID interactions loaded:", f"{len(iid):,}")

print("\nIID columns used:")
for col in iid.columns:
    print(" -", col)


# ---------------------------------------------------------------------
# 4. Normalize gene symbols
# ---------------------------------------------------------------------

string["gene_A_norm"] = (
    string["preferredName_A"]
    .astype(str)
    .str.strip()
    .str.upper()
)

string["gene_B_norm"] = (
    string["preferredName_B"]
    .astype(str)
    .str.strip()
    .str.upper()
)

iid["gene_1_norm"] = (
    iid["symbol1"]
    .astype(str)
    .str.strip()
    .str.upper()
)

iid["gene_2_norm"] = (
    iid["symbol2"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ---------------------------------------------------------------------
# 5. Build UNDIRECTED interaction keys
#
# Example:
# MERTK + GAS6 -> GAS6||MERTK
# GAS6 + MERTK -> GAS6||MERTK
# ---------------------------------------------------------------------

def make_key(a, b):
    a = str(a).strip().upper()
    b = str(b).strip().upper()
    return "||".join(sorted([a, b]))


string["interaction_key"] = [
    make_key(a, b)
    for a, b in zip(
        string["gene_A_norm"],
        string["gene_B_norm"]
    )
]

iid["interaction_key"] = [
    make_key(a, b)
    for a, b in zip(
        iid["gene_1_norm"],
        iid["gene_2_norm"]
    )
]

print(
    "\nUnique STRING interaction keys:",
    f"{string['interaction_key'].nunique():,}"
)

print(
    "Unique IID interaction keys:",
    f"{iid['interaction_key'].nunique():,}"
)

# STRING should already have 9,835 unique undirected edges
assert string["interaction_key"].nunique() == len(string), (
    "STRING table contains repeated undirected interaction keys."
)


# ---------------------------------------------------------------------
# 6. Check duplicate IID records
#
# IID can contain multiple records/evidence lines for the same pair.
# ---------------------------------------------------------------------

iid_duplicate_rows = iid["interaction_key"].duplicated().sum()

print(
    "\nAdditional IID rows sharing an interaction key:",
    f"{iid_duplicate_rows:,}"
)


# ---------------------------------------------------------------------
# 7. Evidence aggregation helper functions
# ---------------------------------------------------------------------

def combine_unique(series):
    """
    Combine unique pipe-separated text values into one pipe-separated field.
    """
    values = []

    for value in series.dropna():

        text = str(value).strip()

        if text in ("", "-", "nan"):
            continue

        for item in text.split("|"):
            item = item.strip()

            if item and item not in values:
                values.append(item)

    return "|".join(values)


def numeric_max(series):
    """
    Return maximum numeric evidence count across IID rows for one pair.
    """
    values = pd.to_numeric(series, errors="coerce")

    if values.notna().any():
        return values.max()

    return pd.NA


# ---------------------------------------------------------------------
# 8. Aggregate IID evidence to ONE row per undirected interaction
# ---------------------------------------------------------------------

print("\nAggregating IID evidence to one row per interaction...")

iid_agg = (
    iid
    .groupby("interaction_key", as_index=False)
    .agg(
        iid_uniprot1=("uniprot1", combine_unique),
        iid_uniprot2=("uniprot2", combine_unique),

        iid_symbol1=("symbol1", combine_unique),
        iid_symbol2=("symbol2", combine_unique),

        iid_methods=("methods", combine_unique),
        iid_pmids=("pmids", combine_unique),
        iid_source_databases=("db_with_ppi", combine_unique),
        iid_evidence_type=("evidence_type", combine_unique),
        iid_detection_type=("detection_type", combine_unique),

        iid_n_exp_methods=("n_exp_methods", numeric_max),
        iid_n_exp_pmids=("n_exp_pmids", numeric_max),
        iid_n_pred_pmids=("n_pred_pmids", numeric_max),
        iid_n_pmids=("n_pmids", numeric_max)
    )
)

print(
    "Aggregated IID interaction records:",
    f"{len(iid_agg):,}"
)


# ---------------------------------------------------------------------
# 9. Merge IID evidence onto the STRING table
#
# LEFT JOIN means every STRING edge is preserved.
# ---------------------------------------------------------------------

annotated = string.merge(
    iid_agg,
    on="interaction_key",
    how="left",
    validate="one_to_one"
)

assert len(annotated) == len(string), (
    "ERROR: STRING interaction count changed during IID merge."
)


# ---------------------------------------------------------------------
# 10. Flag whether an exact IID interaction match was found
# ---------------------------------------------------------------------

annotated["iid_found"] = (
    annotated["iid_evidence_type"].notna()
)


# ---------------------------------------------------------------------
# 11. Separate IID evidence types
# ---------------------------------------------------------------------

def contains_evidence(value, evidence_name):

    if pd.isna(value):
        return False

    evidence = {
        x.strip().lower()
        for x in str(value).split("|")
        if x.strip()
    }

    return evidence_name.lower() in evidence


annotated["iid_experimental"] = (
    annotated["iid_evidence_type"]
    .apply(lambda x: contains_evidence(x, "exp"))
)

annotated["iid_predicted"] = (
    annotated["iid_evidence_type"]
    .apply(lambda x: contains_evidence(x, "pred"))
)

annotated["iid_orthology"] = (
    annotated["iid_evidence_type"]
    .apply(lambda x: contains_evidence(x, "ortho"))
)


# ---------------------------------------------------------------------
# 12. Create readable IID support classification
# ---------------------------------------------------------------------

def classify_iid(row):

    if not row["iid_found"]:
        return "NOT_FOUND"

    if row["iid_experimental"]:
        return "EXPERIMENTAL"

    if row["iid_predicted"] and row["iid_orthology"]:
        return "PREDICTED_AND_ORTHOLOGY"

    if row["iid_predicted"]:
        return "PREDICTED_ONLY"

    if row["iid_orthology"]:
        return "ORTHOLOGY_ONLY"

    return "OTHER_IID_EVIDENCE"


annotated["iid_support_status"] = annotated.apply(
    classify_iid,
    axis=1
)


# ---------------------------------------------------------------------
# 13. Add reproducibility metadata
# ---------------------------------------------------------------------

annotated["iid_database"] = IID_DATABASE
annotated["iid_access_date"] = IID_ACCESS_DATE


# ---------------------------------------------------------------------
# 14. Overall summary
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("IID ANNOTATION RESULTS")
print("=" * 72)

n_found = int(annotated["iid_found"].sum())
n_not_found = len(annotated) - n_found

print(f"\nOriginal STRING interactions : {len(string):,}")
print(f"Final annotated interactions : {len(annotated):,}")
print(f"IID-supported interactions   : {n_found:,}")
print(f"No IID exact match           : {n_not_found:,}")

print(
    f"IID coverage of STRING edges : "
    f"{annotated['iid_found'].mean() * 100:.2f}%"
)

print("\nIID support classification:")

print(
    annotated["iid_support_status"]
    .value_counts()
)

print("\nPercentages:")

print(
    (
        annotated["iid_support_status"]
        .value_counts(normalize=True) * 100
    ).round(2)
)


# ---------------------------------------------------------------------
# 15. Experimental evidence summary
# ---------------------------------------------------------------------

print("\nIID evidence flags:")

print(
    "Experimental:",
    f"{annotated['iid_experimental'].sum():,}"
)

print(
    "Predicted:",
    f"{annotated['iid_predicted'].sum():,}"
)

print(
    "Orthology:",
    f"{annotated['iid_orthology'].sum():,}"
)


# ---------------------------------------------------------------------
# 16. MANUAL QC: MERTK — PROS1
#
# From the manual IID website test, this pair was not directly retrieved.
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("MANUAL QC — MERTK / PROS1")
print("=" * 72)

qc1 = annotated[
    (
        (annotated["gene_A_norm"] == "MERTK") &
        (annotated["gene_B_norm"] == "PROS1")
    )
    |
    (
        (annotated["gene_A_norm"] == "PROS1") &
        (annotated["gene_B_norm"] == "MERTK")
    )
]

display(
    qc1[
        [
            "preferredName_A",
            "preferredName_B",
            "score",
            "iid_found",
            "iid_support_status",
            "iid_evidence_type",
            "iid_methods",
            "iid_pmids",
            "iid_source_databases",
            "iid_n_exp_methods",
            "iid_n_exp_pmids",
            "iid_detection_type"
        ]
    ]
)


# ---------------------------------------------------------------------
# 17. MANUAL QC: MERTK — GAS6
#
# Our small IID test showed experimental + predicted evidence.
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("MANUAL QC — MERTK / GAS6")
print("=" * 72)

qc2 = annotated[
    (
        (annotated["gene_A_norm"] == "MERTK") &
        (annotated["gene_B_norm"] == "GAS6")
    )
    |
    (
        (annotated["gene_A_norm"] == "GAS6") &
        (annotated["gene_B_norm"] == "MERTK")
    )
]

display(
    qc2[
        [
            "preferredName_A",
            "preferredName_B",
            "score",
            "iid_found",
            "iid_support_status",
            "iid_evidence_type",
            "iid_methods",
            "iid_pmids",
            "iid_source_databases",
            "iid_n_exp_methods",
            "iid_n_exp_pmids",
            "iid_detection_type"
        ]
    ]
)


# ---------------------------------------------------------------------
# 18. MANUAL QC: CNGB1 — CNGA1
#
# Our small IID test showed predicted / orthology evidence.
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("MANUAL QC — CNGB1 / CNGA1")
print("=" * 72)

qc3 = annotated[
    (
        (annotated["gene_A_norm"] == "CNGB1") &
        (annotated["gene_B_norm"] == "CNGA1")
    )
    |
    (
        (annotated["gene_A_norm"] == "CNGA1") &
        (annotated["gene_B_norm"] == "CNGB1")
    )
]

display(
    qc3[
        [
            "preferredName_A",
            "preferredName_B",
            "score",
            "iid_found",
            "iid_support_status",
            "iid_evidence_type",
            "iid_methods",
            "iid_pmids",
            "iid_source_databases",
            "iid_n_exp_methods",
            "iid_n_exp_pmids",
            "iid_detection_type"
        ]
    ]
)


# ---------------------------------------------------------------------
# 19. Show representative IID-supported STRING interactions
# ---------------------------------------------------------------------

print("\nRepresentative IID-supported interactions:")

display(
    annotated[
        annotated["iid_found"]
    ][
        [
            "preferredName_A",
            "preferredName_B",
            "score",
            "iid_support_status",
            "iid_evidence_type",
            "iid_methods",
            "iid_pmids",
            "iid_source_databases",
            "iid_n_exp_methods",
            "iid_n_exp_pmids"
        ]
    ].head(20)
)


# ---------------------------------------------------------------------
# 20. Save final IID annotation
# ---------------------------------------------------------------------

annotated.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False
)

print("\n" + "=" * 72)
print("SAVED")
print("=" * 72)

print(OUTPUT_FILE)

print("\nIMPORTANT:")
print("✓ All 9,835 original STRING interactions were retained.")
print("✓ IID was added as independent interaction evidence.")
print("✓ No interaction was removed because IID evidence was absent.")
print("✓ Next step: integrate IID into the COMPARTMENTS + HPA master table.")

STAGE 5C — IID INTERACTION ANNOTATION

STRING file:
/content/rp_network/data/processed/ppi_interactions.tsv

IID file:
/content/rp_network/data/raw/human_annotated_PPIs.txt.gz

STRING interactions loaded: 9,835

Loading IID dataset...
IID interactions loaded: 1,675,927

IID columns used:
 - uniprot1
 - uniprot2
 - symbol1
 - symbol2
 - methods
 - pmids
 - db_with_ppi
 - evidence_type
 - n_exp_methods
 - n_exp_pmids
 - n_pred_pmids
 - n_pmids
 - detection_type

Unique STRING interaction keys: 9,835
Unique IID interaction keys: 1,667,954

Additional IID rows sharing an interaction key: 7,973

Aggregating IID evidence to one row per interaction...
Aggregated IID interaction records: 1,667,954

IID ANNOTATION RESULTS

Original STRING interactions : 9,835
Final annotated interactions : 9,835
IID-supported interactions   : 2,293
No IID exact match           : 7,542
IID coverage of STRING edges : 23.31%

IID support classification:
iid_support_status
NOT_FOUND                  7542
EXPERIMENT

,preferredName_A,preferredName_B,score,iid_found,iid_support_status,iid_evidence_type,iid_methods,iid_pmids,iid_source_databases,iid_n_exp_methods,iid_n_exp_pmids,iid_detection_type
0,MERTK,PROS1,0.999,False,NOT_FOUND,NaN,NaN,NaN,NaN,NaN,NaN,NaN



MANUAL QC — MERTK / GAS6


,preferredName_A,preferredName_B,score,iid_found,iid_support_status,iid_evidence_type,iid_methods,iid_pmids,iid_source_databases,iid_n_exp_methods,iid_n_exp_pmids,iid_detection_type
1,MERTK,GAS6,0.999,True,EXPERIMENTAL,exp|pred,pull down|surface plasmon resonance|pred_FpCla...,8939948|9160883|pred_doi:10.1007/978-3-540-730...,bci|biogrid|hprd|iid|innatedb,2.0,2.0,other|pairwise



MANUAL QC — CNGB1 / CNGA1


,preferredName_A,preferredName_B,score,iid_found,iid_support_status,iid_evidence_type,iid_methods,iid_pmids,iid_source_databases,iid_n_exp_methods,iid_n_exp_pmids,iid_detection_type
6,CNGB1,CNGA1,0.999,True,PREDICTED_AND_ORTHOLOGY,ortho|pred,pred_naiveBayes|pred_PrePPI|pred_RandomForest,pred_23023127|pred_21836163|pred_16082366,iid,0.0,0.0,



Representative IID-supported interactions:


,preferredName_A,preferredName_B,score,iid_support_status,iid_evidence_type,iid_methods,iid_pmids,iid_source_databases,iid_n_exp_methods,iid_n_exp_pmids
1,MERTK,GAS6,0.999,EXPERIMENTAL,exp|pred,pull down|surface plasmon resonance|pred_FpCla...,8939948|9160883|pred_doi:10.1007/978-3-540-730...,bci|biogrid|hprd|iid|innatedb,2.0,2.0
2,MVK,HMGCR,0.999,PREDICTED_ONLY,pred,pred_FpClass|pred_PrePPI|pred_RandomForest,pred_23023127|pred_25402006|pred_21836163,iid,0.0,0.0
3,MVK,MVD,0.999,PREDICTED_ONLY,pred,pred_FpClass|pred_naiveBayes|pred_PrePPI,pred_23023127|pred_25402006|pred_16082366,iid,0.0,0.0
6,CNGB1,CNGA1,0.999,PREDICTED_AND_ORTHOLOGY,ortho|pred,pred_naiveBayes|pred_PrePPI|pred_RandomForest,pred_23023127|pred_21836163|pred_16082366,iid,0.0,0.0
7,CNGB1,CNGA2,0.999,PREDICTED_ONLY,pred,pred_naiveBayes|pred_RandomForest,pred_21836163|pred_16082366,iid,0.0,0.0
8,PRPF8,LSM8,0.999,EXPERIMENTAL,exp|pred,affinity chromatography technology|pred_Random...,28515276|pred_21836163,biogrid|iid,1.0,1.0
9,PRPF8,CDC40,0.999,EXPERIMENTAL,exp|ortho|pred,biochemical|bioid|pred_RandomForest,26344197|34079125|pred_21836163,biogrid|iid,2.0,2.0
10,PRPF8,DDX23,0.999,EXPERIMENTAL,exp|ortho|pred,affinity chromatography technology|anti tag co...,26186194|26344197|26414014|28514442|28515276|3...,biogrid|iid|intact,5.0,11.0
11,PRPF8,EFTUD2,0.999,EXPERIMENTAL,exp|ortho|pred,affinity chromatography technology|anti tag co...,16723661|17317632|22939629|25263594|26344197|2...,biogrid|iid|intact,11.0,18.0
12,PRPF8,TXNL4A,0.999,EXPERIMENTAL,exp|pred,affinity chromatography technology|anti tag co...,28515276|33961781|35271311|pred_21836163,biogrid|iid|intact,2.0,3.0



SAVED
/content/rp_network/data/processed/iid_interaction_annotation.tsv

IMPORTANT:
✓ All 9,835 original STRING interactions were retained.
✓ IID was added as independent interaction evidence.
✓ No interaction was removed because IID evidence was absent.
✓ Next step: integrate IID into the COMPARTMENTS + HPA master table.


In [ ]:
# =====================================================================
# NOTEBOOK 05 — STAGE 5D
# MASTER CONTEXT-EVIDENCE TABLE
#
# Combines:
#   1. STRING network
#   2. COMPARTMENTS annotation (threshold >= 2)
#   3. HPA retina expression (nTPM > 0)
#   4. IID interaction evidence
#
# IMPORTANT:
# No interactions are removed here.
# This creates the evidence table that TissueNet will be added to next.
# =====================================================================

import pandas as pd
from pathlib import Path

BASE = Path("/content/rp_network")
PROC = BASE / "data/processed"

COMP_FILE = PROC / "compartments_interaction_annotation_t2.tsv"
HPA_FILE  = PROC / "hpa_retina_interaction_annotation.tsv"
IID_FILE  = PROC / "iid_interaction_annotation.tsv"

OUTPUT = PROC / "context_specific_evidence_master.tsv"

print("=" * 72)
print("STAGE 5D — BUILD MASTER CONTEXT-EVIDENCE TABLE")
print("=" * 72)


# ---------------------------------------------------------------------
# 1. Check files
# ---------------------------------------------------------------------

for f in [COMP_FILE, HPA_FILE, IID_FILE]:
    assert f.exists(), f"Missing file: {f}"
    print("Found:", f.name)


# ---------------------------------------------------------------------
# 2. Load completed annotation tables
# ---------------------------------------------------------------------

comp = pd.read_csv(COMP_FILE, sep="\t", low_memory=False)
hpa  = pd.read_csv(HPA_FILE, sep="\t", low_memory=False)
iid  = pd.read_csv(IID_FILE, sep="\t", low_memory=False)

print("\nRows loaded:")
print("COMPARTMENTS :", f"{len(comp):,}")
print("HPA          :", f"{len(hpa):,}")
print("IID          :", f"{len(iid):,}")

assert len(comp) == 9835
assert len(hpa)  == 9835
assert len(iid)  == 9835


# ---------------------------------------------------------------------
# 3. Create identical undirected interaction keys
# ---------------------------------------------------------------------

def make_key(a, b):
    a = str(a).strip().upper()
    b = str(b).strip().upper()
    return "||".join(sorted([a, b]))


comp["interaction_key"] = [
    make_key(a, b)
    for a, b in zip(comp["protein_A"], comp["protein_B"])
]

hpa["interaction_key"] = [
    make_key(a, b)
    for a, b in zip(hpa["protein_A"], hpa["protein_B"])
]

# IID already has interaction_key, but rebuild it from the original
# STRING names for consistency.
iid["interaction_key"] = [
    make_key(a, b)
    for a, b in zip(
        iid["preferredName_A"],
        iid["preferredName_B"]
    )
]


# ---------------------------------------------------------------------
# 4. Verify unique interaction identities
# ---------------------------------------------------------------------

assert comp["interaction_key"].nunique() == 9835
assert hpa["interaction_key"].nunique() == 9835
assert iid["interaction_key"].nunique() == 9835

assert set(comp["interaction_key"]) == set(hpa["interaction_key"])
assert set(comp["interaction_key"]) == set(iid["interaction_key"])

print("\n✓ All three tables describe the same 9,835 interactions.")


# ---------------------------------------------------------------------
# 5. Start master table with COMPARTMENTS
# ---------------------------------------------------------------------

master = comp.copy()


# ---------------------------------------------------------------------
# 6. Add selected HPA columns
# ---------------------------------------------------------------------

hpa_keep = [
    "interaction_key",
    "retina_nTPM_A",
    "retina_nTPM_B",
    "A_in_HPA_retina",
    "B_in_HPA_retina",
    "expression_status_gt0"
]

missing_hpa = [c for c in hpa_keep if c not in hpa.columns]

if missing_hpa:
    print("\nAvailable HPA columns:")
    print(hpa.columns.tolist())
    raise ValueError(
        f"Expected HPA columns missing: {missing_hpa}"
    )

master = master.merge(
    hpa[hpa_keep],
    on="interaction_key",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------------------
# 7. Add selected IID columns
# ---------------------------------------------------------------------

iid_keep = [
    "interaction_key",
    "iid_found",
    "iid_support_status",
    "iid_evidence_type",
    "iid_experimental",
    "iid_predicted",
    "iid_orthology",
    "iid_methods",
    "iid_pmids",
    "iid_source_databases",
    "iid_n_exp_methods",
    "iid_n_exp_pmids",
    "iid_n_pred_pmids",
    "iid_n_pmids",
    "iid_detection_type",
    "iid_database",
    "iid_access_date"
]

missing_iid = [c for c in iid_keep if c not in iid.columns]

if missing_iid:
    print("\nAvailable IID columns:")
    print(iid.columns.tolist())
    raise ValueError(
        f"Expected IID columns missing: {missing_iid}"
    )

master = master.merge(
    iid[iid_keep],
    on="interaction_key",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------------------
# 8. Confirm nothing was lost
# ---------------------------------------------------------------------

assert len(master) == 9835

print("\n✓ All 9,835 interactions retained.")


# ---------------------------------------------------------------------
# 9. Create simple evidence flags
# ---------------------------------------------------------------------

master["compartments_supported"] = (
    master["compartment_status"] == "PASS"
)

master["hpa_retina_supported"] = (
    master["expression_status_gt0"] == "BOTH_EXPRESSED"
)

master["iid_supported"] = (
    master["iid_found"] == True
)


# ---------------------------------------------------------------------
# 10. Count completed evidence layers
#
# NOTE:
# This is an annotation score/count, NOT a biological probability.
# ---------------------------------------------------------------------

master["n_context_evidence_layers"] = (
    master[
        [
            "compartments_supported",
            "hpa_retina_supported",
            "iid_supported"
        ]
    ]
    .astype(int)
    .sum(axis=1)
)


# ---------------------------------------------------------------------
# 11. Create descriptive context category
# ---------------------------------------------------------------------

def context_category(row):

    comp_ok = row["compartments_supported"]
    hpa_ok  = row["hpa_retina_supported"]
    iid_ok  = row["iid_supported"]

    if comp_ok and hpa_ok and iid_ok:
        return "SUPPORTED_BY_COMPARTMENTS_HPA_IID"

    if comp_ok and hpa_ok:
        return "SUPPORTED_BY_COMPARTMENTS_HPA"

    if comp_ok and iid_ok:
        return "SUPPORTED_BY_COMPARTMENTS_IID"

    if hpa_ok and iid_ok:
        return "SUPPORTED_BY_HPA_IID"

    if comp_ok:
        return "COMPARTMENTS_ONLY"

    if hpa_ok:
        return "HPA_ONLY"

    if iid_ok:
        return "IID_ONLY"

    return "NO_POSITIVE_EVIDENCE_IN_CURRENT_LAYERS"


master["current_context_category"] = master.apply(
    context_category,
    axis=1
)


# ---------------------------------------------------------------------
# 12. Summary
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("MASTER TABLE SUMMARY")
print("=" * 72)

print("\nInteractions:", f"{len(master):,}")

print(
    "COMPARTMENTS-supported:",
    f"{master['compartments_supported'].sum():,}"
)

print(
    "HPA retina-supported:",
    f"{master['hpa_retina_supported'].sum():,}"
)

print(
    "IID-supported:",
    f"{master['iid_supported'].sum():,}"
)

print("\nNumber of positive evidence layers:")

print(
    master["n_context_evidence_layers"]
    .value_counts()
    .sort_index()
)

print("\nCurrent context categories:")

print(
    master["current_context_category"]
    .value_counts()
)


# ---------------------------------------------------------------------
# 13. QC — MERTK/GAS6
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("QC — MERTK / GAS6")
print("=" * 72)

qc = master[
    master["interaction_key"] == make_key("MERTK", "GAS6")
]

display(
    qc[
        [
            "protein_A",
            "protein_B",
            "string_score",
            "compartment_status",
            "shared_compartments",
            "retina_nTPM_A",
            "retina_nTPM_B",
            "expression_status_gt0",
            "iid_found",
            "iid_support_status",
            "iid_n_exp_methods",
            "iid_n_exp_pmids",
            "n_context_evidence_layers",
            "current_context_category"
        ]
    ]
)


# ---------------------------------------------------------------------
# 14. Save
# ---------------------------------------------------------------------

master.to_csv(
    OUTPUT,
    sep="\t",
    index=False
)

print("\n" + "=" * 72)
print("SAVED")
print("=" * 72)

print(OUTPUT)

print("\nThis is now the master context-evidence table.")
print("TissueNet evidence will be added to this table next.")

STAGE 5D — BUILD MASTER CONTEXT-EVIDENCE TABLE
Found: compartments_interaction_annotation_t2.tsv
Found: hpa_retina_interaction_annotation.tsv
Found: iid_interaction_annotation.tsv

Rows loaded:
COMPARTMENTS : 9,835
HPA          : 9,835
IID          : 9,835

✓ All three tables describe the same 9,835 interactions.

✓ All 9,835 interactions retained.

MASTER TABLE SUMMARY

Interactions: 9,835
COMPARTMENTS-supported: 9,751
HPA retina-supported: 9,202
IID-supported: 2,293

Number of positive evidence layers:
n_context_evidence_layers
0      30
1     604
2    6961
3    2240
Name: count, dtype: int64

Current context categories:
current_context_category
SUPPORTED_BY_COMPARTMENTS_HPA             6909
SUPPORTED_BY_COMPARTMENTS_HPA_IID         2240
COMPARTMENTS_ONLY                          551
HPA_ONLY                                    52
SUPPORTED_BY_COMPARTMENTS_IID               51
NO_POSITIVE_EVIDENCE_IN_CURRENT_LAYERS      30
IID_ONLY                                     1
SUPPORTED_BY_HP

,protein_A,protein_B,string_score,compartment_status,shared_compartments,retina_nTPM_A,retina_nTPM_B,expression_status_gt0,iid_found,iid_support_status,iid_n_exp_methods,iid_n_exp_pmids,n_context_evidence_layers,current_context_category
1,MERTK,GAS6,0.999,PASS,Anchoring junction; Apoptotic body; Cell junct...,15.4,5.8,BOTH_EXPRESSED,True,EXPERIMENTAL,2.0,2.0,3,SUPPORTED_BY_COMPARTMENTS_HPA_IID



SAVED
/content/rp_network/data/processed/context_specific_evidence_master.tsv

This is now the master context-evidence table.
TissueNet evidence will be added to this table next.


In [ ]:
# Save all processed files to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)
print("✓ All files saved to Drive.")
print()
print("Files to commit to GitHub:")
print("  docs/progress_log.md")
print("  docs/database_sources.md")
print("  data/processed/compartments_interaction_annotation_t2.tsv")
print("  data/processed/hpa_retina_interaction_annotation.tsv")
print("  data/processed/iid_interaction_annotation.tsv")
print("  data/processed/context_specific_evidence_master.tsv")

✓ All files saved to Drive.

Files to commit to GitHub:
  docs/progress_log.md
  docs/database_sources.md
  data/processed/compartments_interaction_annotation_t2.tsv
  data/processed/hpa_retina_interaction_annotation.tsv
  data/processed/iid_interaction_annotation.tsv
  data/processed/context_specific_evidence_master.tsv


In [ ]:
from getpass import getpass

github_username = "ahmedyomiisiaka"
github_token = getpass("Paste your GitHub token here (hidden): ")

Paste your GitHub token here (hidden): ··········


In [32]:
!git push origin main

Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 2.76 MiB | 2.53 MiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ahmedyomiisiaka/RP-Network-Analysis.git
   3875526..694cc22  main -> main
